# ✅ Competition-Ready Tunix Notebook (Error-Resistant) — “Show Your Work”
**Goal:** Fine-tune **Gemma2-2B-IT** (or Gemma3 1B) with **Tunix + GRPO** to reliably output:

```xml
<reasoning>...</reasoning>
<answer>...</answer>
```

This notebook is written to satisfy the competition’s **winning factors**:
- **Notebook quality (35 pts):** clear data, hyperparams, prompt, training strategy, techniques
- **Model quality (45 pts):** trains in a single session, saves loadable checkpoints, correct output format
- **Video (20 pts):** includes a script outline + visuals checklist
- **Optional (15 pts):** multi-session checkpoint resume instructions

> Important: No notebook can guarantee “winning”, but this is designed to be **clean, reproducible, and stable**.

Generated: **2025-12-31**


## 0) Kaggle Runtime Setup (TPU vs GPU)
### If TPU is disabled in Kaggle UI:
Even if you have TPU quota, Kaggle sometimes disables TPUs due to capacity.
✅ Use **GPU P100** or **GPU T4** and this notebook will still work.

### Required Kaggle Settings
- **Internet: ON** (needed for KaggleHub + TFDS downloads)
- **Accelerator:** TPU v5e-8 OR GPU P100/T4

After changing accelerator: **Restart session**


## 1) Install (Kaggle)
If Kaggle already has these packages, you can skip this cell.

In [ ]:
# Please delete this cell after use Cell 0 — Kaggle-safe Tunix setup (keeps Kaggle JAX/JAXLIB)

!pip -q uninstall -y google-tunix tunix flax orbax-checkpoint 2>/dev/null || true

# Install Tunix WITHOUT deps so it doesn't try to upgrade JAX/JAXLIB
!pip -q install --no-deps "google-tunix[prod]==0.1.3"

# Install Tunix expected deps (since we used --no-deps)
!pip -q install jaxtyping grain qwix tensorboardX optax kagglehub tqdm humanize

# IMPORTANT: Match Flax/Orbax to Kaggle's JAX 0.5.x
!pip -q install "flax==0.8.4" "orbax-checkpoint==0.6.4"

# (Optional) If your notebook uses datasets/tokenizers
!pip -q install transformers datasets

# Cell 0: Kaggle-safe installs (keep Kaggle's JAX/JAXLIB) 
!pip -q uninstall -y flax orbax-checkpoint tunix google-tunix 2>/dev/null || true # Install Tunix 
!pip -q install --no-deps "google-tunix[prod]==0.1.3" # Install the missing deps Tunix expects (because we used --no-deps) 
!pip -q install -q qwix tensorboardX # Match Flax to Kaggle JAX 0.8.x (prevents abstracted_axes / jit issues) 
!pip -q install "flax==0.12.2" # Orbax version that works with Tunix setups on Kaggle
!pip -q install "orbax-checkpoint==0.11.31" # Other deps
!pip -q install optax kagglehub grain tensorflow tensorflow_datasets tqdm humanize

import os
os._exit(0)  # force clean restart


/usr/lib/python3.12/pty.py:95: RuntimeWarning: os.fork() was called. os.fork() is incompatible with multithreaded code, and JAX is multithreaded, so this will likely lead to a deadlock.
  pid, fd = os.forkpty()


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dopamine-rl 4.1.2 requires gymnasium>=1.0.0, but you have gymnasium 0.29.0 which is incompatible.
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-tunix 0.1.3 requires flax>=0.11.1, but you have flax 0.8.4 which is incompatible.
qwix 0.1.5 requires flax>=0.12.0, but you have flax 0.8.4 which is incompatible.
dopamine-rl 4.1.2 requires gymnasium>=1.0.0, but you have gymnasium 0.29.0 which is incompatible.
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dopamine-rl 4.1.2 requires gymnasium>=1.0.0, but you have gymnasium 0.29.0 which is incompatible.


In [ ]:
# Cell 0 — Kaggle JAX 0.5.3 lane using older google-tunix

!pip -q uninstall -y tunix google-tunix flax orbax-checkpoint qwix tensorflow tensorflow_datasets 2>/dev/null || true

# Try an older Tunix that is compatible with JAX 0.5.x (most stable approach)
!pip -q install --no-deps "google-tunix[prod]==0.1.1"

# Install deps (minimal)
!pip -q install jaxtyping grain tensorboardX optax kagglehub tqdm humanize

# JAX 0.5.x compatible Flax/Orbax
!pip -q install "flax==0.8.4" "orbax-checkpoint==0.6.4"

import os
os._exit(0)


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 232.8/232.8 kB 6.0 MB/s eta 0:00:00a 0:00:01
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dopamine-rl 4.1.2 requires tensorflow>=2.2.0, which is not installed.
dopamine-rl 4.1.2 requires gymnasium>=1.0.0, but you have gymnasium 0.29.0 which is incompatible.


In [3]:
import jax
print("jax:", jax.__version__, "jaxlib:", jax.lib.__version__)
print("backend:", jax.default_backend())
print("devices:", jax.devices())

import jax, flax, orbax.checkpoint as ocp
import tunix
print("jax:", jax.__version__)
print("flax:", flax.__version__)
print("orbax:", ocp.__version__)
print("tunix:", tunix.__version__)


import jax
print("jax:", jax.__version__, "jaxlib:", jax.lib.__version__)
print("devices:", jax.devices())

import tunix
print("tunix imported OK:", tunix.__version__)


ERROR:2026-01-01 22:20:00,631:jax._src.xla_bridge:475: Jax plugin configuration error: Exception when calling jax_plugins.xla_cuda12.initialize()
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/jax/_src/xla_bridge.py", line 473, in discover_pjrt_plugins
    plugin_module.initialize()
  File "/usr/local/lib/python3.12/dist-packages/jax_plugins/xla_cuda12/__init__.py", line 328, in initialize
    _check_cuda_versions(raise_on_first_error=True)
  File "/usr/local/lib/python3.12/dist-packages/jax_plugins/xla_cuda12/__init__.py", line 150, in _check_cuda_versions
    assert cuda_versions is not None
           ^^^^^^^^^^^^^^^^^^^^^^^^^
AssertionError


jax: 0.8.2 jaxlib: 0.8.2
backend: cpu
devices: [CpuDevice(id=0)]


/usr/local/lib/python3.12/dist-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'repr' attribute with value False was provided to the `Field()` function, which has no effect in the context it was used. 'repr' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` statement was used, or if the `Field()` function was attached to a single member of a union type.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'frozen' attribute with value True was provided to the `Field()` function, which has no effect in the context it was used. 'frozen' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` 

jax: 0.8.2
flax: 0.12.2
orbax: 0.11.31
tunix: 0.1.3
jax: 0.8.2 jaxlib: 0.8.2
devices: [CpuDevice(id=0)]
tunix imported OK: 0.1.3


In [4]:
import jax
print("backend:", jax.default_backend())
print("devices:", jax.devices())



backend: cpu
devices: [CpuDevice(id=0)]


In [5]:
import jax, flax
import orbax.checkpoint as ocp
import tunix

print("✅ Tunix stack OK")
print("jax:", jax.__version__, "| jaxlib:", jax.lib.__version__)
print("flax:", flax.__version__)
print("orbax-checkpoint:", getattr(ocp, "__version__", "ok"))
print("devices:", jax.devices())


✅ Tunix stack OK
jax: 0.8.2 | jaxlib: 0.8.2
flax: 0.12.2
orbax-checkpoint: 0.11.31
devices: [CpuDevice(id=0)]


## 2) Imports + Environment Checks

In [6]:
import os, gc, re, math, json, time
from dataclasses import dataclass
from typing import Any, Dict, List, Tuple, Callable

import jax
import jax.numpy as jnp
import optax
import tensorflow_datasets as tfds
from tqdm.auto import tqdm

import kagglehub
import grain
from orbax import checkpoint as ocp
from flax import nnx

from tunix.generate import sampler as sampler_lib
from tunix.generate import tokenizer_adapter as tokenizer_lib
from tunix.models.gemma import model as gemma_lib
from tunix.models.gemma import params as params_lib
from tunix.rl import rl_cluster as rl_cluster_lib
from tunix.rl.rollout import base_rollout
from tunix.rl.grpo.grpo_learner import GRPOConfig, GRPOLearner

print("JAX devices:", jax.devices())
print("Default backend:", jax.default_backend())


JAX devices: [CpuDevice(id=0)]
Default backend: cpu


## 3) Configuration (safe defaults + GPU/TPU auto-tuning)

In [7]:
# =========================
# Cell 3 (STEP 3): Prompt + Config (Judge-impressive + Stable)
# =========================

from dataclasses import dataclass
import jax

# ===== Required format tokens =====
REASONING_START = "<reasoning>"
REASONING_END   = "</reasoning>"
ANSWER_START    = "<answer>"
ANSWER_END      = "</answer>"

# ✅ Judge-friendly: strict formatting + self-check + concise final answer
SYSTEM_PROMPT = f"""
You are a careful assistant trained to "show your work" consistently.

CRITICAL OUTPUT FORMAT (must be exact):
1) First write {REASONING_START} ... {REASONING_END}
2) Then write {ANSWER_START} ... {ANSWER_END}
3) Do NOT output anything else (no greetings, no extra lines, no markdown).

REASONING RULES:
- Use short, numbered steps.
- Keep reasoning minimal but complete.
- If the task is creative, explain decisions briefly and clearly.

ANSWER RULES:
- Keep {ANSWER_START} concise.
- If the answer is a single value, output only that value.
- If a short sentence is needed, keep it to one sentence.

FINAL SELF-CHECK before you finish:
- Did you include exactly one {REASONING_START} and one {ANSWER_START} block?
- Did you avoid any text outside those blocks?
""".strip()

# ✅ Correct chat template: system -> user -> model
TEMPLATE = """<start_of_turn>system
{system_prompt}<end_of_turn>
<start_of_turn>user
{question}<end_of_turn>
<start_of_turn>model
"""

# ✅ Wrap every task to reinforce the contract (helps format compliance a lot)
def wrap_question(q: str) -> str:
    return (
        "Follow the required format exactly.\n"
        f"Return ONLY:\n{REASONING_START}...{REASONING_END}\n{ANSWER_START}...{ANSWER_END}\n\n"
        f"Task:\n{q}"
    )

@dataclass
class Cfg:
    # Model (Gemma2-2B-IT)
    model_kagglehub_id: str = "google/gemma-2/flax/gemma2-2b-it/1"
    params_subdir: str = "gemma2-2b-it"   # KaggleHub layout

    # Generation constraints (competition: <1K output tokens is fine)
    max_prompt_len: int = 384
    max_gen_tokens: int = 512
    temperature: float = 0.85
    top_k: int = 50
    top_p: float = 0.95

    # GRPO
    num_generations: int = 4
    num_iterations: int = 1
    beta: float = 0.10
    epsilon: float = 0.2

    # Training
    train_micro_bs: int = 2
    mini_bs: int = 8
    rollout_micro_bs: int = 8
    lr: float = 2e-6
    max_steps: int = 1500
    eval_every: int = 250

    # Parallelism
    tp_size: int = 4   # TPU typically 4 or 8; GPU should be 1

    # Checkpoints
    ckpt_dir: str = "/kaggle/working/ckpts"
    save_every: int = 500
    max_to_keep: int = 3

CFG = Cfg()

# Auto-tune for GPU (reduces OOM risk)
backend = jax.default_backend()
if backend == "gpu":
    CFG.tp_size = 1
    CFG.num_generations = 2
    CFG.max_steps = 900
    CFG.max_gen_tokens = 320
    CFG.rollout_micro_bs = 4
    CFG.temperature = 0.80
    CFG.top_p = 0.95

print("Final CFG:", CFG)

# Quick sanity check
_example = wrap_question("If Mary has 3 apples and buys 2 more, how many apples does she have?")
_prompt = TEMPLATE.format(system_prompt=SYSTEM_PROMPT, question=_example)
print("\n--- Prompt preview (first 500 chars) ---\n", _prompt[:500])


Final CFG: Cfg(model_kagglehub_id='google/gemma-2/flax/gemma2-2b-it/1', params_subdir='gemma2-2b-it', max_prompt_len=384, max_gen_tokens=512, temperature=0.85, top_k=50, top_p=0.95, num_generations=4, num_iterations=1, beta=0.1, epsilon=0.2, train_micro_bs=2, mini_bs=8, rollout_micro_bs=8, lr=2e-06, max_steps=1500, eval_every=250, tp_size=4, ckpt_dir='/kaggle/working/ckpts', save_every=500, max_to_keep=3)

--- Prompt preview (first 500 chars) ---
 <start_of_turn>system
You are a careful assistant trained to "show your work" consistently.

CRITICAL OUTPUT FORMAT (must be exact):
1) First write <reasoning> ... </reasoning>
2) Then write <answer> ... </answer>
3) Do NOT output anything else (no greetings, no extra lines, no markdown).

REASONING RULES:
- Use short, numbered steps.
- Keep reasoning minimal but complete.
- If the task is creative, explain decisions briefly and clearly.

ANSWER RULES:
- Keep <answer> concise.
- If the answer is


## 4) Download Gemma checkpoint + tokenizer (KaggleHub)

In [8]:
import os, glob

CKPT_ROOT = kagglehub.model_download(CFG.model_kagglehub_id)
print("CKPT_ROOT:", CKPT_ROOT)

TOKENIZER_PATH = os.path.join(CKPT_ROOT, "tokenizer.model")
PARAMS_DIR = os.path.join(CKPT_ROOT, CFG.params_subdir)

assert os.path.exists(TOKENIZER_PATH), f"Missing tokenizer.model at {TOKENIZER_PATH}"
assert os.path.isdir(PARAMS_DIR), f"Missing params dir at {PARAMS_DIR}"

# (Optional) show what is inside params dir (proves correct checkpoint layout)
print("Tokenizer:", TOKENIZER_PATH)
print("Params dir:", PARAMS_DIR)
print("Params dir sample files:", sorted(glob.glob(os.path.join(PARAMS_DIR, '*'))[:8]))



CKPT_ROOT: /kaggle/input/gemma-2/flax/gemma2-2b-it/1
Tokenizer: /kaggle/input/gemma-2/flax/gemma2-2b-it/1/tokenizer.model
Params dir: /kaggle/input/gemma-2/flax/gemma2-2b-it/1/gemma2-2b-it
Params dir sample files: ['/kaggle/input/gemma-2/flax/gemma2-2b-it/1/gemma2-2b-it/_CHECKPOINT_METADATA', '/kaggle/input/gemma-2/flax/gemma2-2b-it/1/gemma2-2b-it/_METADATA', '/kaggle/input/gemma-2/flax/gemma2-2b-it/1/gemma2-2b-it/checkpoint', '/kaggle/input/gemma-2/flax/gemma2-2b-it/1/gemma2-2b-it/d', '/kaggle/input/gemma-2/flax/gemma2-2b-it/1/gemma2-2b-it/descriptor', '/kaggle/input/gemma-2/flax/gemma2-2b-it/1/gemma2-2b-it/manifest.ocdbt', '/kaggle/input/gemma-2/flax/gemma2-2b-it/1/gemma2-2b-it/ocdbt.process_0']


In [9]:
!pip -q uninstall -y jax_plugins jax-cuda12-plugin jax_cuda12_plugin nvidia-jax-cuda12-plugin || true

import pkgutil
mods = [m.name for m in pkgutil.iter_modules()]
hits = [m for m in mods if ("jax_plugins" in m) or ("xla_cuda12" in m) or ("cuda12" in m)]
print("Found related modules:", hits[:50])




/usr/lib/python3.12/pty.py:95: RuntimeWarning: os.fork() was called. os.fork() is incompatible with multithreaded code, and JAX is multithreaded, so this will likely lead to a deadlock.
  pid, fd = os.forkpty()


Found related modules: []


In [10]:
import os

# Force JAX to never try CUDA/TPU plugins
os.environ["JAX_PLATFORMS"] = "cpu"
os.environ["JAX_PLATFORM_NAME"] = "cpu"   # extra safety
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"

print("✅ Set JAX to CPU-only. Now restart kernel and run Cell B next.")


✅ Set JAX to CPU-only. Now restart kernel and run Cell B next.


In [11]:
import jax
print("jax:", jax.__version__, "jaxlib:", jax.lib.__version__)
print("backend:", jax.default_backend())
print("devices:", jax.devices())


jax: 0.8.2 jaxlib: 0.8.2
backend: cpu
devices: [CpuDevice(id=0)]


In [12]:
import os
os.environ["JAX_PLATFORMS"] = "cpu"                 # force CPU backend
os.environ["JAX_PJRT_CLIENT_CREATE_OPTIONS"] = ""   # avoid plugin auto-config


In [13]:
import jax
print(jax.__version__, jax.lib.__version__)
print(jax.devices())


0.8.2 0.8.2
[CpuDevice(id=0)]


## 5) Build Mesh + Load models (CPU-safe)

In [14]:
# ============================================================
# Step 5) Build Mesh + Load Tokenizer + Load Params + Build Models
# - Self-contained: re-creates CKPT_ROOT if missing
# - GUARANTEES: tokenizer, ref_model, actor_model exist
# - Works on CPU-only or GPU (if available)
# ============================================================

import os, gc
import jax
import kagglehub

from tunix.generate import tokenizer_adapter as tokenizer_lib
from tunix.models.gemma import model as gemma_lib
from tunix.models.gemma import params as params_lib

# ---- Ensure CFG exists (minimal fallback if Step 3 wasn't run) ----
if "CFG" not in globals():
    from dataclasses import dataclass

    @dataclass
    class Cfg:
        model_kagglehub_id: str = "google/gemma-2/flax/gemma2-2b-it/1"
        params_subdir: str = "gemma2-2b-it"
        ckpt_dir: str = "/kaggle/working/ckpts"
        tp_size: int = 1

    CFG = Cfg()
    print("⚠️ CFG was missing, created a minimal default CFG.")

# ---- Ensure CKPT_ROOT exists (re-run Step 4 automatically) ----
if "CKPT_ROOT" not in globals():
    CKPT_ROOT = kagglehub.model_download(CFG.model_kagglehub_id)
    print("⚠️ CKPT_ROOT was missing, re-downloaded via kagglehub.")
print("CKPT_ROOT:", CKPT_ROOT)

# ---- Paths ----
TOKENIZER_PATH = os.path.join(CKPT_ROOT, "tokenizer.model")
PARAMS_DIR = os.path.join(CKPT_ROOT, getattr(CFG, "params_subdir", "gemma2-2b-it"))

assert os.path.exists(TOKENIZER_PATH), f"Missing tokenizer.model at {TOKENIZER_PATH}"
assert os.path.exists(PARAMS_DIR), f"Missing params dir at {PARAMS_DIR}"

# ---- Runtime info ----
print("Backend:", jax.default_backend())
print("Devices:", jax.devices())

# ---- Mesh (safe defaults) ----
tp = 1
mesh = jax.make_mesh(
    (1, tp),
    ("fsdp", "tp"),
    axis_types=(jax.sharding.AxisType.Auto, jax.sharding.AxisType.Auto),
)
print("Mesh:", mesh)

# ---- Tokenizer ----
tokenizer = tokenizer_lib.Tokenizer(tokenizer_path=TOKENIZER_PATH)
print("✅ Tokenizer loaded:", TOKENIZER_PATH)

# ---- Load params ----
params = params_lib.load_and_format_params(PARAMS_DIR)
print("✅ Params loaded. Top keys:", list(params.keys()))

# ---- Build models ----
ref_model = gemma_lib.Transformer.from_params(params, version="2-2b-it")
actor_model = gemma_lib.Transformer.from_params(params, version="2-2b-it")

# Cleanup
del params
gc.collect()

print("✅ Models loaded.")
print("Globals check:",
      "tokenizer" in globals(),
      "ref_model" in globals(),
      "actor_model" in globals())


CKPT_ROOT: /kaggle/input/gemma-2/flax/gemma2-2b-it/1
Backend: cpu
Devices: [CpuDevice(id=0)]
Mesh: Mesh('fsdp': 1, 'tp': 1, axis_types=(Auto, Auto))
✅ Tokenizer loaded: /kaggle/input/gemma-2/flax/gemma2-2b-it/1/tokenizer.model
✅ Params loaded. Top keys: ['transformer']
✅ Models loaded.
Globals check: True True True


## 6) Data (Training + Testing)
We use **GSM8K** for verifiable math + a small cross-domain set for generalization.

In [3]:
# =========================
# Cell: Build GSM8K + Extra Tasks (self-contained, error-resistant)
# =========================
import os
import tensorflow_datasets as tfds
import grain

from dataclasses import dataclass

# ---- Required format tokens ----
REASONING_START = "<reasoning>"
REASONING_END   = "</reasoning>"
ANSWER_START    = "<answer>"
ANSWER_END      = "</answer>"

SYSTEM_PROMPT = f"""You are a helpful assistant.
Always respond in this exact format:
{REASONING_START}...{REASONING_END}
{ANSWER_START}...{ANSWER_END}
Rules:
- Do NOT write anything outside the tags.
- Put all step-by-step work inside <reasoning>.
- For math problems, put ONLY the final number in <answer> (no units, no words).
- Keep <answer> concise.
"""

TEMPLATE = """<start_of_turn>system
{system_prompt}<end_of_turn>
<start_of_turn>user
{question}<end_of_turn>
<start_of_turn>model
"""

@dataclass
class Cfg:
    train_micro_bs: int = 2

CFG = Cfg()

def extract_hash_answer(text: str):
    # GSM8K answers typically look like: "... #### 42"
    if "####" not in text:
        return None
    return text.split("####", 1)[1].strip()

def build_gsm8k(split: str, batch_size: int, data_dir: str):
    import tensorflow_datasets.text.gsm8k  # ensures builder is registered

    ds = tfds.data_source(
        "gsm8k",
        split=split,
        data_dir=data_dir,
        builder_kwargs={"file_format": tfds.core.FileFormat.ARRAY_RECORD},
        download=True,
    )

    def _as_text(v):
        return v if isinstance(v, str) else v.decode("utf-8")

    return (
        grain.MapDataset.source(ds)
        .shuffle(seed=42)
        .map(lambda x: {
            "prompts": TEMPLATE.format(
                system_prompt=SYSTEM_PROMPT,
                question=_as_text(x["question"]),
            ),
            "question": _as_text(x["question"]),
            "answer": extract_hash_answer(_as_text(x["answer"])),
        })
        .batch(batch_size)
    )

EXTRA_TASKS = [
    ("Summarize in 2 sentences: A transformer uses attention to model relationships in text.", None),
    ("Write Python code for Fibonacci(n). Mention time complexity.", None),
    ("Explain why the sky is blue in simple terms.", None),
    ("Give 5 creative product ideas for a chess-themed birthday party.", None),
    ("Explain what overfitting is and how to reduce it.", None),
]

def build_extra(batch_size: int):
    rows = [{
        "prompts": TEMPLATE.format(system_prompt=SYSTEM_PROMPT, question=q),
        "question": q,
        "answer": a
    } for q, a in EXTRA_TASKS]
    return grain.MapDataset.source(rows).shuffle(seed=7).batch(batch_size)

TRAIN_DIR = "/kaggle/working/data/train"
TEST_DIR  = "/kaggle/working/data/test"

os.makedirs(TRAIN_DIR, exist_ok=True)
os.makedirs(TEST_DIR, exist_ok=True)

train_gsm8k = build_gsm8k("train", CFG.train_micro_bs, TRAIN_DIR)
test_gsm8k  = build_gsm8k("test",  CFG.train_micro_bs, TEST_DIR)
extra_ds    = build_extra(CFG.train_micro_bs)

# Preview one batch
b = next(iter(train_gsm8k[:1]))
print("Batch keys:", b.keys())
print("Prompt snippet:", b["prompts"][0][:200], "...")
print("Answer:", b["answer"][0])


Batch keys: dict_keys(['answer', 'prompts', 'question'])
Prompt snippet: <start_of_turn>system
You are a helpful assistant.
Always respond in this exact format:
<reasoning>...</reasoning>
<answer>...</answer>
Rules:
- Do NOT write anything outside the tags.
- Put all step- ...
Answer: 13


In [4]:
import tensorflow_datasets as tfds
tfds.core.utils.gcs_utils._is_gcs_disabled = True  # avoids some slow GCS checks on Kaggle


In [6]:
# ============================================================
# Cell 6.5) Preflight Guard (NO generate_one dependency)
# - Confirms runtime + required globals exist before Sampler
# ============================================================
import jax

print("JAX backend:", jax.default_backend())
print("Devices:", jax.devices())

required = ["CFG", "TOKENIZER_PATH", "PARAMS_DIR", "tokenizer", "actor_model", "TEMPLATE", "SYSTEM_PROMPT"]
missing = [k for k in required if k not in globals()]
if missing:
    raise NameError(f"❌ Missing required objects (run earlier cells): {missing}")

# Optional: downshift CFG before building sampler (prevents long compiles)
CFG.max_prompt_len = int(getattr(CFG, "max_prompt_len", 256))
CFG.max_gen_tokens = int(getattr(CFG, "max_gen_tokens", 384))

print("✅ Preflight OK")
print("CFG.max_prompt_len:", CFG.max_prompt_len)
print("CFG.max_gen_tokens:", CFG.max_gen_tokens)



JAX backend: cpu
Devices: [CpuDevice(id=0)]
✅ Preflight OK
CFG.max_prompt_len: 256
CFG.max_gen_tokens: 384


## 7) Sampler (for baseline + eval)

In [7]:
# ============================================================
# Cell 7) Sampler (Baseline + Eval) — CPU-safe, non-hanging
# ============================================================
import re, time
from typing import Dict, Tuple
import jax
from tunix.generate import sampler as sampler_lib

# ---- Detect backend and auto-downshift on CPU ----
BACKEND = jax.default_backend()
IS_CPU = (BACKEND == "cpu")

MAX_PROMPT_LENGTH = int(getattr(CFG, "max_prompt_len", 256))
TOTAL_GENERATION_STEPS = int(getattr(CFG, "max_gen_tokens", 384))
TEMPERATURE = float(getattr(CFG, "temperature", 0.7))
TOP_K = int(getattr(CFG, "top_k", 50))
TOP_P = float(getattr(CFG, "top_p", 0.95))

# If CPU, reduce generation length for responsiveness
if IS_CPU:
    TOTAL_GENERATION_STEPS = min(TOTAL_GENERATION_STEPS, 64)
    MAX_PROMPT_LENGTH = min(MAX_PROMPT_LENGTH, 128)

CACHE_SIZE = int(MAX_PROMPT_LENGTH + TOTAL_GENERATION_STEPS + 256)

model_config = gemma_lib.ModelConfig.gemma2_2b()

sampler = sampler_lib.Sampler(
    transformer=actor_model,
    tokenizer=tokenizer,
    cache_config=sampler_lib.CacheConfig(
        cache_size=CACHE_SIZE,
        num_layers=int(model_config.num_layers),
        num_kv_heads=int(model_config.num_kv_heads),
        head_dim=int(model_config.head_dim),
    ),
)

print("✅ Sampler ready")
print("Backend:", BACKEND)
print(f"PROMPT_LEN={MAX_PROMPT_LENGTH} | GEN_TOKENS={TOTAL_GENERATION_STEPS} | CACHE_SIZE={CACHE_SIZE}")

# -------------------------
# Format parsing utilities
# -------------------------
_REASON_RE = re.compile(r"<reasoning>(.*?)</reasoning>", re.DOTALL | re.IGNORECASE)
_ANSWER_RE = re.compile(r"<answer>(.*?)</answer>", re.DOTALL | re.IGNORECASE)

def parse_reasoning_answer(text: str) -> Tuple[str | None, str | None]:
    r = _REASON_RE.search(text)
    a = _ANSWER_RE.search(text)
    reasoning = (r.group(1).strip() if r else None)
    answer = (a.group(1).strip() if a else None)
    return reasoning, answer

def strict_format_ok(text: str) -> bool:
    r, a = parse_reasoning_answer(text)
    return (r is not None) and (a is not None)

def _first_number(s: str | None):
    if not s:
        return None
    s = s.replace(",", "")
    m = re.search(r"-?\d+(\.\d+)?", s)
    return float(m.group(0)) if m else None

def gsm8k_numeric_match(pred_ans: str | None, gold_ans: str | None) -> bool:
    pn = _first_number(pred_ans)
    gn = _first_number(gold_ans)
    return (pn is not None and gn is not None and abs(pn - gn) < 1e-9)

# -------------------------
# Generation (with soft timeout)
# -------------------------
def generate_one(
    question: str,
    temperature: float = TEMPERATURE,
    top_k: int = TOP_K,
    top_p: float = TOP_P,
    seed: int = 0,
    max_steps: int | None = None,
    soft_timeout_s: float = 120.0,  # fail fast if it truly stalls
) -> str:
    prompt = TEMPLATE.format(system_prompt=SYSTEM_PROMPT, question=question)
    steps = int(max_steps if max_steps is not None else TOTAL_GENERATION_STEPS)

    t0 = time.time()
    out = sampler(
        input_strings=[prompt],
        max_generation_steps=steps,
        temperature=float(temperature),
        top_k=int(top_k),
        top_p=float(top_p),
        echo=False,
        seed=int(seed),
    )
    if (time.time() - t0) > soft_timeout_s:
        return "<reasoning>timeout</reasoning><answer>timeout</answer>"
    return out.text[0]

# -------------------------
# Baseline eval (CPU-safe default)
# -------------------------
def run_baseline_eval(dataset, n_samples: int | None = None, log_every: int = 2) -> Dict[str, float]:
    if n_samples is None:
        n_samples = 8 if IS_CPU else 32

    t0 = time.time()
    total = 0
    fmt_ok = 0
    num_ok = 0

    it = iter(dataset)
    while total < n_samples:
        batch = next(it)
        bs = len(batch["question"])
        for i in range(bs):
            if total >= n_samples:
                break

            q = batch["question"][i]
            gold = batch["answer"][i] if "answer" in batch else None

            # short decode during eval to keep it fast
            pred_text = generate_one(q, seed=total, max_steps=(32 if IS_CPU else None))
            ok_fmt = strict_format_ok(pred_text)
            if ok_fmt:
                fmt_ok += 1

            _, pred_ans = parse_reasoning_answer(pred_text)
            if gold is not None and gsm8k_numeric_match(pred_ans, gold):
                num_ok += 1

            total += 1
            if total % log_every == 0:
                print(f"… evaluated {total}/{n_samples}")

    dt = time.time() - t0
    fmt_rate = fmt_ok / max(total, 1)
    num_rate = num_ok / max(total, 1)

    print("\n✅ Baseline results")
    print(f"Samples: {total} | Time: {dt:.1f}s")
    print(f"Strict format compliance: {fmt_rate*100:.1f}%")
    print(f"GSM8K numeric-match:      {num_rate*100:.1f}%")

    return {
        "samples": float(total),
        "seconds": float(dt),
        "format_rate": float(fmt_rate),
        "gsm8k_numeric": float(num_rate),
    }

# ---- Micro sanity (32 tokens only) ----
print("\n--- Micro sanity (fast) ---")
demo = generate_one("2+2=?", seed=0, max_steps=32, soft_timeout_s=120.0)
print(demo[:300])

print("\n--- Baseline eval (CPU-safe) ---")
baseline_metrics = run_baseline_eval(test_gsm8k, n_samples=None, log_every=2)
baseline_metrics


✅ Sampler ready
Backend: cpu
PROMPT_LEN=128 | GEN_TOKENS=64 | CACHE_SIZE=448

--- Micro sanity (fast) ---
<reasoning>
2 + 2 = 4
<answer>4</answer> 
<end_of_turn>

--- Baseline eval (CPU-safe) ---
… evaluated 2/8
… evaluated 4/8
… evaluated 6/8
… evaluated 8/8

✅ Baseline results
Samples: 8 | Time: 354.3s
Strict format compliance: 0.0%
GSM8K numeric-match:      0.0%


{'samples': 8.0,
 'seconds': 354.27371740341187,
 'format_rate': 0.0,
 'gsm8k_numeric': 0.0}

In [9]:
import jax; print(jax.default_backend(), jax.devices())


cpu [CpuDevice(id=0)]


In [10]:
# ============================================================
# Cell 8) GRPO Train (SAFE SMOKE) — Tunix-version compatible
# Fix: GRPOLearner(...) does NOT accept algo_config in your build.
# This cell auto-detects the right constructor signature.
# ============================================================
import os, re, time, inspect
import optax
import jax

from tunix.rl import rl_cluster as rl_cluster_lib
from tunix.rl.rollout import base_rollout
from tunix.rl.grpo.grpo_learner import GRPOConfig, GRPOLearner

# ---- Detect backend ----
BACKEND = jax.default_backend()
print("Backend:", BACKEND, "| Devices:", jax.devices())

# ---- Check required globals ----
missing = [k for k in ["CFG", "mesh", "tokenizer", "actor_model", "ref_model", "train_gsm8k", "test_gsm8k"] if k not in globals()]
if missing:
    raise NameError(f"Missing required globals: {missing}. Run earlier cells that create them.")

CKPT_DIR = str(getattr(CFG, "ckpt_dir", "/kaggle/working/ckpts"))
os.makedirs(CKPT_DIR, exist_ok=True)

# ---- Start from CFG (safe fallbacks) ----
MAX_PROMPT_LENGTH = int(getattr(CFG, "max_prompt_len", 128))
TOTAL_GENERATION_STEPS = int(getattr(CFG, "max_gen_tokens", 32))
TEMPERATURE = float(getattr(CFG, "temperature", 0.7))
TOP_K = int(getattr(CFG, "top_k", 50))
TOP_P = float(getattr(CFG, "top_p", 0.95))

NUM_GENERATIONS = int(getattr(CFG, "num_generations", 2))  # GRPO must be >1
NUM_ITERATIONS  = int(getattr(CFG, "num_iterations", 1))
BETA            = float(getattr(CFG, "beta", 0.08))
EPSILON         = float(getattr(CFG, "epsilon", 0.2))

MAX_STEPS = int(getattr(CFG, "max_steps", 2))             # SMOKE ONLY
EVAL_EVERY_N_STEPS = int(getattr(CFG, "eval_every", 1))
TRAIN_MICRO_BATCH_SIZE = int(getattr(CFG, "train_micro_bs", 1))
LR = float(getattr(CFG, "lr", 2e-6))

# ---- CPU hard overrides ----
if BACKEND == "cpu":
    MAX_PROMPT_LENGTH = 64
    TOTAL_GENERATION_STEPS = 16
    NUM_GENERATIONS = 2           # ✅ minimum allowed for GRPO
    MAX_STEPS = 1
    TRAIN_MICRO_BATCH_SIZE = 1
    EVAL_EVERY_N_STEPS = 1
    print("⚠️ CPU mode: forcing tiny rollout/training to avoid long runs.")

print("SMOKE CONFIG:",
      "prompt_len=", MAX_PROMPT_LENGTH,
      "gen_steps=", TOTAL_GENERATION_STEPS,
      "num_gen=", NUM_GENERATIONS,
      "max_steps=", MAX_STEPS,
      "micro_bs=", TRAIN_MICRO_BATCH_SIZE)

# ---- Reward fns ----
_REASON_RE = re.compile(r"<reasoning>(.*?)</reasoning>", re.DOTALL | re.IGNORECASE)
_ANSWER_RE = re.compile(r"<answer>(.*?)</answer>", re.DOTALL | re.IGNORECASE)

def _parse(text: str):
    r = _REASON_RE.search(text)
    a = _ANSWER_RE.search(text)
    return (r.group(1).strip() if r else None), (a.group(1).strip() if a else None)

def _first_number(s):
    if not s: return None
    s = s.replace(",", "")
    m = re.search(r"-?\d+(\.\d+)?", s)
    return float(m.group(0)) if m else None

def reward_format_exact(prompt, completion, example=None):
    r, a = _parse(completion)
    return 1.0 if (r is not None and a is not None) else 0.0

def reward_answer_numeric(prompt, completion, example=None):
    if not example or example.get("answer") is None:
        return 0.0
    _, pred_ans = _parse(completion)
    pn = _first_number(pred_ans)
    gn = _first_number(example["answer"])
    return 1.0 if (pn is not None and gn is not None and abs(pn - gn) < 1e-9) else 0.0

reward_fns = [reward_format_exact, reward_answer_numeric]

# ---- Optimizer ----
optimizer = optax.adamw(learning_rate=LR)

# ---- EOS tokens (robust) ----
EOS_TOKENS = []
for attr in ["eos_id", "eos_token_id"]:
    if hasattr(tokenizer, attr):
        v = getattr(tokenizer, attr)
        v = v() if callable(v) else v
        try:
            EOS_TOKENS = [int(v)]
            break
        except Exception:
            pass
if not EOS_TOKENS:
    EOS_TOKENS = [1]
print("EOS_TOKENS:", EOS_TOKENS)

# ---- Cluster config ----
cluster_config = rl_cluster_lib.ClusterConfig(
    role_to_mesh={
        rl_cluster_lib.Role.ACTOR: mesh,
        rl_cluster_lib.Role.REFERENCE: mesh,
        rl_cluster_lib.Role.ROLLOUT: mesh,
    },
    rollout_engine="vanilla",
    offload_to_cpu=False,
    training_config=rl_cluster_lib.RLTrainingConfig(
        actor_optimizer=optimizer,
        eval_every_n_steps=EVAL_EVERY_N_STEPS,
        max_steps=MAX_STEPS,
        mini_batch_size=TRAIN_MICRO_BATCH_SIZE,
        train_micro_batch_size=TRAIN_MICRO_BATCH_SIZE,
        checkpoint_root_directory=CKPT_DIR,
    ),
    rollout_config=base_rollout.RolloutConfig(
        max_tokens_to_generate=TOTAL_GENERATION_STEPS,
        max_prompt_length=MAX_PROMPT_LENGTH,
        kv_cache_size=MAX_PROMPT_LENGTH + TOTAL_GENERATION_STEPS + 128,
        temperature=TEMPERATURE,
        top_p=TOP_P,
        top_k=TOP_K,
        eos_tokens=EOS_TOKENS,
    ),
)

# ---- GRPO config ----
grpo_config = GRPOConfig(
    num_generations=NUM_GENERATIONS,
    num_iterations=NUM_ITERATIONS,
    beta=BETA,
    epsilon=EPSILON,
)

# ---- RL cluster ----
rl_cluster = rl_cluster_lib.RLCluster(
    actor=actor_model,
    reference=ref_model,
    tokenizer=tokenizer,
    cluster_config=cluster_config,
)

# ---- Build trainer (signature-safe) ----
sig = str(inspect.signature(GRPOLearner))
print("GRPOLearner signature:", sig)

kwargs = {"rl_cluster": rl_cluster, "reward_fns": reward_fns}

trainer = None
try:
    # Most common in Tunix: grpo_config=
    trainer = GRPOLearner(**kwargs, grpo_config=grpo_config)
    print("✅ GRPOLearner constructed with grpo_config=")
except TypeError:
    try:
        # Some builds use config=
        trainer = GRPOLearner(**kwargs, config=grpo_config)
        print("✅ GRPOLearner constructed with config=")
    except TypeError:
        # Fallback: positional (rl_cluster, reward_fns, grpo_config)
        trainer = GRPOLearner(rl_cluster, reward_fns, grpo_config)
        print("✅ GRPOLearner constructed with positional args")

print("✅ GRPO trainer ready. Starting SMOKE train()...")
print("Checkpoint dir:", CKPT_DIR)

t0 = time.time()
trainer.train(train_gsm8k, test_gsm8k)
dt = time.time() - t0

print(f"✅ SMOKE training finished in {dt:.1f}s")
print("✅ Checkpoints (if any) under:", CKPT_DIR)


Backend: cpu | Devices: [CpuDevice(id=0)]
⚠️ CPU mode: forcing tiny rollout/training to avoid long runs.
SMOKE CONFIG: prompt_len= 64 gen_steps= 16 num_gen= 2 max_steps= 1 micro_bs= 1
EOS_TOKENS: [1]


GRPOLearner signature: (rl_cluster: 'rl_cluster_lib.RLCluster', reward_fns: 'RewardFn | List[RewardFn]', grpo_config: 'GRPOConfig', metric_fns: 'Sequence[MetricFn] | None' = None, data_shuffle_seed: 'int | None' = None)


✅ GRPOLearner constructed with grpo_config=
✅ GRPO trainer ready. Starting SMOKE train()...
Checkpoint dir: /kaggle/working/ckpts


ValueError: Total sampling steps 272 must be less than the cache size 208.

## 8) Rewards (fast, deterministic)
Format + reasoning style + math correctness when labels exist.

In [20]:
import re

# If you already defined these earlier, this won't hurt.
REASONING_START = "<reasoning>"
REASONING_END   = "</reasoning>"
ANSWER_START    = "<answer>"
ANSWER_END      = "</answer>"


In [22]:
FORMAT_RE = re.compile(
    rf"^\s*{re.escape(REASONING_START)}.+?{re.escape(REASONING_END)}.*?"
    rf"{re.escape(ANSWER_START)}.+?{re.escape(ANSWER_END)}\s*$",
    flags=re.DOTALL,
)
ANSWER_RE = re.compile(
    rf"{re.escape(ANSWER_START)}\s*(.+?)\s*{re.escape(ANSWER_END)}",
    flags=re.DOTALL,
)

# Number extracted ONLY from inside <answer> ... </answer>
NUM_RE = re.compile(
    rf"{re.escape(ANSWER_START)}\s*([-+]?\d[\d,]*\.?\d*)\s*{re.escape(ANSWER_END)}",
    flags=re.DOTALL,
)

def r_format_exact(prompts, completions, **kwargs):
    return [1.5 if FORMAT_RE.search(c) else -0.5 for c in completions]

def r_format_soft(prompts, completions, **kwargs):
    scores = []
    for c in completions:
        s = 0.0
        s += 0.5 if c.count(REASONING_START) == 1 else -0.5
        s += 0.5 if c.count(REASONING_END)   == 1 else -0.5
        s += 0.5 if c.count(ANSWER_START)    == 1 else -0.5
        s += 0.5 if c.count(ANSWER_END)      == 1 else -0.5
        scores.append(s)
    return scores

def r_reasoning_quality(prompts, completions, **kwargs):
    scores = []
    for c in completions:
        m = ANSWER_RE.search(c)
        ans = m.group(1).strip() if m else ""

        rs = c.find(REASONING_START); re_ = c.find(REASONING_END)
        reasoning = c[rs+len(REASONING_START):re_].strip() if (rs!=-1 and re_!=-1 and re_>rs) else ""

        s = 0.0
        if 80 <= len(reasoning) <= 900: s += 1.0
        elif 20 <= len(reasoning) < 80: s += 0.3
        else: s -= 0.6

        s += 0.4 if len(ans) <= 140 else -0.4
        if "as an ai" in c.lower(): s -= 0.7
        if ("Step" in reasoning) or ("1)" in reasoning) or ("\n- " in reasoning): s += 0.2

        scores.append(s)
    return scores

def r_math_correct(prompts, completions, answer, **kwargs):
    scores = []
    for c, a in zip(completions, answer):
        if a is None:
            scores.append(0.0)
            continue

        m = NUM_RE.search(c)
        if not m:
            scores.append(-0.2)
            continue

        g = m.group(1).replace(",", "").strip()
        try:
            gf = float(g)
            af = float(a.replace(",", "").strip())

            # handle gold == 0 safely
            if abs(af) < 1e-12:
                scores.append(2.0 if abs(gf) < 1e-9 else -0.5)
                continue

            # close-match scoring
            if abs(gf - af) < 1e-9:
                scores.append(2.0)
            else:
                ratio = gf / af
                if 0.95 <= ratio <= 1.05: scores.append(0.6)
                elif 0.90 <= ratio <= 1.10: scores.append(0.3)
                else: scores.append(-0.5)
        except Exception:
            scores.append(-0.3)

    return scores


## 9) RLCluster + GRPO Trainer (correct configs)

In [25]:
# ============================================================
# Cell 9) SELF-CONTAINED: RLCluster + GRPO Trainer (auto-fix)
# - Creates CFG if missing
# - Finds CKPT_ROOT if missing
# - Loads tokenizer + params + builds ref_model/actor_model if missing
# - Builds mesh if missing
# - Defines reward fns if missing
# - Builds optimizer + training_cfg + rollout_cfg + cluster_cfg
# - Builds RLCluster + GRPOConfig + GRPO Trainer (correct signature)
# NOTE: This cell DOES NOT start training (no .train()).
# ============================================================

import os, re, gc, time, inspect
from dataclasses import dataclass

import jax
import jax.numpy as jnp

# ---- Tunix imports ----
from tunix.generate import tokenizer_adapter as tokenizer_lib
from tunix.models.gemma import model as gemma_lib
from tunix.models.gemma import params as params_lib

from tunix.rl import rl_cluster as rl_cluster_lib
from tunix.rl.rollout import base_rollout
from tunix.rl.grpo.grpo_learner import GRPOConfig, GRPOLearner

# ---- Optional deps used in configs ----
import optax
import orbax.checkpoint as ocp

print("Backend:", jax.default_backend(), "| Devices:", jax.devices())

# ============================================================
# 1) Ensure CFG exists (minimal defaults)
# ============================================================
if "CFG" not in globals():
    @dataclass
    class Cfg:
        # model location
        model_kagglehub_id: str = "gemma-2/flax/gemma2-2b-it/1"
        params_subdir: str = "gemma2-2b-it"   # under CKPT_ROOT
        ckpt_dir: str = "/kaggle/working/ckpts"

        # gen
        max_prompt_len: int = 128
        max_gen_tokens: int = 64
        temperature: float = 0.7
        top_k: int = 50
        top_p: float = 0.95

        # training
        lr: float = 2e-6
        eval_every: int = 50
        max_steps: int = 200
        mini_bs: int = 1
        train_micro_bs: int = 1
        rollout_micro_bs: int = 1
        save_every: int = 50
        max_to_keep: int = 2

        # GRPO algo (MUST be > 1)
        num_generations: int = 4
        num_iterations: int = 1
        beta: float = 0.08
        epsilon: float = 0.2

    CFG = Cfg()
    print("✅ CFG created (auto).")

# ============================================================
# 2) Ensure CKPT_ROOT exists
# ============================================================
if "CKPT_ROOT" not in globals():
    # Try common Kaggle input path first (works when you already have the dataset attached)
    candidate = "/kaggle/input/gemma-2/flax/gemma2-2b-it/1"
    if os.path.exists(candidate):
        CKPT_ROOT = candidate
        print("✅ CKPT_ROOT auto-detected:", CKPT_ROOT)
    else:
        # Fallback: try kagglehub if available
        try:
            import kagglehub
            CKPT_ROOT = kagglehub.model_download(CFG.model_kagglehub_id)
            print("✅ CKPT_ROOT downloaded via kagglehub:", CKPT_ROOT)
        except Exception as e:
            raise NameError(
                "❌ CKPT_ROOT not found and kagglehub download failed.\n"
                "Attach the Gemma2 model dataset in Kaggle or install/use kagglehub.\n"
                f"Error: {e}"
            )

TOKENIZER_PATH = os.path.join(CKPT_ROOT, "tokenizer.model")
PARAMS_DIR = os.path.join(CKPT_ROOT, getattr(CFG, "params_subdir", "gemma2-2b-it"))

assert os.path.exists(TOKENIZER_PATH), f"Missing tokenizer.model at {TOKENIZER_PATH}"
assert os.path.exists(PARAMS_DIR), f"Missing params dir at {PARAMS_DIR}"

# ============================================================
# 3) Ensure mesh exists
# ============================================================
if "mesh" not in globals():
    tp = 1
    mesh = jax.make_mesh(
        (1, tp),
        ("fsdp", "tp"),
        axis_types=(jax.sharding.AxisType.Auto, jax.sharding.AxisType.Auto),
    )
    print("✅ mesh created:", mesh)
else:
    print("✅ mesh exists:", mesh)

# ============================================================
# 4) Ensure tokenizer exists
# ============================================================
if "tokenizer" not in globals():
    tokenizer = tokenizer_lib.Tokenizer(tokenizer_path=TOKENIZER_PATH)
    print("✅ tokenizer loaded:", TOKENIZER_PATH)
else:
    print("✅ tokenizer exists")

# ============================================================
# 5) Ensure models exist (ref_model + actor_model)
# ============================================================
need_models = ("ref_model" not in globals()) or ("actor_model" not in globals())
if need_models:
    print("Loading params + building models (auto)…")
    params = params_lib.load_and_format_params(PARAMS_DIR)
    ref_model   = gemma_lib.Transformer.from_params(params, version="2-2b-it")
    actor_model = gemma_lib.Transformer.from_params(params, version="2-2b-it")
    del params
    gc.collect()
    print("✅ models built: ref_model + actor_model")
else:
    print("✅ models exist")

# ============================================================
# 6) Ensure reward functions exist (format + soft + reasoning + math)
#    (These match Tunix reward API: (prompts, completions, **kwargs))
# ============================================================
if "re" not in globals():
    import re

REASONING_START = "<reasoning>"
REASONING_END   = "</reasoning>"
ANSWER_START    = "<answer>"
ANSWER_END      = "</answer>"

FORMAT_RE = re.compile(
    rf"^\s*{re.escape(REASONING_START)}.+?{re.escape(REASONING_END)}.*?"
    rf"{re.escape(ANSWER_START)}.+?{re.escape(ANSWER_END)}\s*$",
    flags=re.DOTALL,
)
ANSWER_RE = re.compile(
    rf"{re.escape(ANSWER_START)}\s*(.+?)\s*{re.escape(ANSWER_END)}",
    flags=re.DOTALL,
)
NUM_RE = re.compile(
    rf"{re.escape(ANSWER_START)}\s*([-+]?\d[\d,]*\.?\d*)\s*{re.escape(ANSWER_END)}",
    flags=re.DOTALL,
)

def r_format_exact(prompts, completions, **kwargs):
    return [1.5 if FORMAT_RE.search(c) else -0.5 for c in completions]

def r_format_soft(prompts, completions, **kwargs):
    scores = []
    for c in completions:
        s = 0.0
        s += 0.5 if c.count(REASONING_START) == 1 else -0.5
        s += 0.5 if c.count(REASONING_END)   == 1 else -0.5
        s += 0.5 if c.count(ANSWER_START)    == 1 else -0.5
        s += 0.5 if c.count(ANSWER_END)      == 1 else -0.5
        scores.append(s)
    return scores

def r_reasoning_quality(prompts, completions, **kwargs):
    scores = []
    for c in completions:
        m = ANSWER_RE.search(c)
        ans = m.group(1).strip() if m else ""
        rs = c.find(REASONING_START); re_ = c.find(REASONING_END)
        reasoning = c[rs+len(REASONING_START):re_].strip() if (rs!=-1 and re_!=-1 and re_>rs) else ""

        s = 0.0
        if 80 <= len(reasoning) <= 900: s += 1.0
        elif 20 <= len(reasoning) < 80: s += 0.3
        else: s -= 0.6

        s += 0.4 if len(ans) <= 140 else -0.4
        if "as an ai" in c.lower(): s -= 0.7
        if ("Step" in reasoning) or ("1)" in reasoning) or ("\n- " in reasoning): s += 0.2

        scores.append(s)
    return scores

def r_math_correct(prompts, completions, answer=None, **kwargs):
    # If 'answer' provided by dataset, reward numeric match
    scores = []
    if answer is None:
        return [0.0 for _ in completions]

    for c, a in zip(completions, answer):
        if a is None:
            scores.append(0.0)
            continue
        m = NUM_RE.search(c)
        if not m:
            scores.append(-0.2)
            continue
        g = m.group(1).replace(",", "").strip()
        try:
            gf = float(g)
            af = float(str(a).replace(",", "").strip())
            if abs(af) < 1e-12:
                scores.append(2.0 if abs(gf) < 1e-9 else -0.5)
                continue
            if abs(gf - af) < 1e-9:
                scores.append(2.0)
            else:
                ratio = gf / af
                if 0.95 <= ratio <= 1.05: scores.append(0.6)
                elif 0.90 <= ratio <= 1.10: scores.append(0.3)
                else: scores.append(-0.5)
        except Exception:
            scores.append(-0.3)
    return scores

reward_fns = [r_format_exact, r_format_soft, r_reasoning_quality, r_math_correct]
print("✅ reward functions ready")

# ============================================================
# 7) Build optimizer + training/rollout configs + cluster
# ============================================================
os.makedirs(CFG.ckpt_dir, exist_ok=True)

optimizer = optax.chain(
    optax.clip_by_global_norm(0.1),
    optax.adamw(learning_rate=CFG.lr, b1=0.9, b2=0.99, weight_decay=0.1),
)

training_cfg = rl_cluster_lib.RLTrainingConfig(
    actor_optimizer=optimizer,
    eval_every_n_steps=int(CFG.eval_every),
    max_steps=int(CFG.max_steps),
    mini_batch_size=int(CFG.mini_bs),
    train_micro_batch_size=int(CFG.train_micro_bs),
    rollout_micro_batch_size=int(getattr(CFG, "rollout_micro_bs", CFG.train_micro_bs)),
    checkpoint_root_directory=str(CFG.ckpt_dir),
    checkpointing_options=ocp.CheckpointManagerOptions(
        save_interval_steps=int(CFG.save_every),
        max_to_keep=int(CFG.max_to_keep),
    ),
)

# --- EOS tokens safe fallback ---
EOS_TOKENS = [1]
for attr in ["eos_id", "eos_token_id"]:
    if hasattr(tokenizer, attr):
        v = getattr(tokenizer, attr)
        v = v() if callable(v) else v
        try:
            EOS_TOKENS = [int(v)]
            break
        except Exception:
            pass

rollout_cfg = base_rollout.RolloutConfig(
    max_tokens_to_generate=int(CFG.max_gen_tokens),
    max_prompt_length=int(CFG.max_prompt_len),
    kv_cache_size=int(CFG.max_prompt_len + CFG.max_gen_tokens + 256),
    temperature=float(CFG.temperature),
    top_p=float(CFG.top_p),
    top_k=int(CFG.top_k),
    eos_tokens=EOS_TOKENS,
)

cluster_cfg = rl_cluster_lib.ClusterConfig(
    role_to_mesh={
        rl_cluster_lib.Role.ACTOR: mesh,
        rl_cluster_lib.Role.REFERENCE: mesh,
        rl_cluster_lib.Role.ROLLOUT: mesh,
    },
    rollout_engine="vanilla",
    offload_to_cpu=False,
    training_config=training_cfg,
    rollout_config=rollout_cfg,
)

rl_cluster = rl_cluster_lib.RLCluster(
    actor=actor_model,
    reference=ref_model,
    tokenizer=tokenizer,
    cluster_config=cluster_cfg,
)

# ============================================================
# 8) GRPO config + trainer (signature-safe)
# ============================================================
if int(CFG.num_generations) <= 1:
    print("⚠️ num_generations must be > 1 for GRPO. Forcing to 2.")
    CFG.num_generations = 2

grpo_cfg = GRPOConfig(
    num_generations=int(CFG.num_generations),
    num_iterations=int(CFG.num_iterations),
    beta=float(CFG.beta),
    epsilon=float(CFG.epsilon),
)

# GRPOLearner signature changed across versions; handle both.
sig = inspect.signature(GRPOLearner)
kwargs = {"rl_cluster": rl_cluster, "reward_fns": reward_fns}

if "grpo_config" in sig.parameters:
    kwargs["grpo_config"] = grpo_cfg
elif "algo_config" in sig.parameters:
    kwargs["algo_config"] = grpo_cfg
else:
    # last resort: try positional
    pass

try:
    grpo_trainer = GRPOLearner(**kwargs)
except TypeError:
    # try positional fallback
    grpo_trainer = GRPOLearner(rl_cluster, reward_fns, grpo_cfg)

print("\n✅ Trainer ready.")
print("  CKPT_DIR:", CFG.ckpt_dir)
print("  max_steps:", CFG.max_steps, "| num_generations:", CFG.num_generations, "| num_iterations:", CFG.num_iterations)
print("  prompt_len:", CFG.max_prompt_len, "| gen_tokens:", CFG.max_gen_tokens)
print("  EOS_TOKENS:", EOS_TOKENS)


Backend: cpu | Devices: [CpuDevice(id=0)]
✅ mesh exists: Mesh('fsdp': 1, 'tp': 1, axis_types=(Auto, Auto))
✅ tokenizer exists
✅ models exist
✅ reward functions ready


AttributeError: 'Cfg' object has no attribute 'ckpt_dir'

## 10) Baseline outputs (before training)

CELL 10 A)RECOVERY CELL

In [1]:
# ============================================================
# Cell A) FULL RECOVERY — rebuild CKPT paths + mesh + tokenizer + models
# (Works even after kernel restart; does NOT depend on CFG)
# ============================================================
import os, gc, glob
import jax
from tunix.generate import tokenizer_adapter as tokenizer_lib
from tunix.models.gemma import model as gemma_lib
from tunix.models.gemma import params as params_lib

print("Backend:", jax.default_backend(), "| Devices:", jax.devices())

# ---- 1) Auto-detect CKPT_ROOT (Gemma2 2B IT Kaggle dataset) ----
# Expected path you already saw:
# /kaggle/input/gemma-2/flax/gemma2-2b-it/1
candidates = [
    "/kaggle/input/gemma-2/flax/gemma2-2b-it/1",
    "/kaggle/input/gemma-2/flax/gemma2-2b-it",
]
CKPT_ROOT = None
for c in candidates:
    if os.path.exists(c):
        CKPT_ROOT = c
        break

# fallback: search
if CKPT_ROOT is None:
    hits = glob.glob("/kaggle/input/**/flax/**/1", recursive=True)
    hits = [h for h in hits if "gemma2-2b-it" in h]
    if hits:
        CKPT_ROOT = hits[0]

if CKPT_ROOT is None:
    raise FileNotFoundError("❌ Could not find Gemma2-2B-IT checkpoint under /kaggle/input.")

print("✅ CKPT_ROOT:", CKPT_ROOT)

# ---- 2) Build paths ----
TOKENIZER_PATH = os.path.join(CKPT_ROOT, "tokenizer.model")
if not os.path.exists(TOKENIZER_PATH):
    raise FileNotFoundError(f"❌ tokenizer.model not found at: {TOKENIZER_PATH}")

# Params dir is typically /.../1/gemma2-2b-it
PARAMS_DIR = os.path.join(CKPT_ROOT, "gemma2-2b-it")
if not os.path.exists(PARAMS_DIR):
    # fallback: find any subdir that looks like params
    subdirs = [d for d in glob.glob(os.path.join(CKPT_ROOT, "*")) if os.path.isdir(d)]
    guess = [d for d in subdirs if "gemma" in os.path.basename(d)]
    if guess:
        PARAMS_DIR = guess[0]
    else:
        raise FileNotFoundError(f"❌ params dir not found under: {CKPT_ROOT}")

print("✅ TOKENIZER_PATH:", TOKENIZER_PATH)
print("✅ PARAMS_DIR:", PARAMS_DIR)

# ---- 3) Mesh (single-host safe) ----
tp = 1
mesh = jax.make_mesh(
    (1, tp),
    ("fsdp", "tp"),
    axis_types=(jax.sharding.AxisType.Auto, jax.sharding.AxisType.Auto),
)
print("✅ mesh:", mesh)

# ---- 4) Tokenizer ----
tokenizer = tokenizer_lib.Tokenizer(tokenizer_path=TOKENIZER_PATH)
print("✅ tokenizer loaded")

# ---- 5) Load params + build models ----
print("Loading params (this can take a bit)...")
params = params_lib.load_and_format_params(PARAMS_DIR)

ref_model   = gemma_lib.Transformer.from_params(params, version="2-2b-it")
actor_model = gemma_lib.Transformer.from_params(params, version="2-2b-it")

del params
gc.collect()

print("✅ models ready:", ("ref_model" in globals()), ("actor_model" in globals()))


/usr/local/lib/python3.12/dist-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'repr' attribute with value False was provided to the `Field()` function, which has no effect in the context it was used. 'repr' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` statement was used, or if the `Field()` function was attached to a single member of a union type.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'frozen' attribute with value True was provided to the `Field()` function, which has no effect in the context it was used. 'frozen' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` 

Backend: cpu | Devices: [CpuDevice(id=0)]
✅ CKPT_ROOT: /kaggle/input/gemma-2/flax/gemma2-2b-it/1
✅ TOKENIZER_PATH: /kaggle/input/gemma-2/flax/gemma2-2b-it/1/tokenizer.model
✅ PARAMS_DIR: /kaggle/input/gemma-2/flax/gemma2-2b-it/1/gemma2-2b-it
✅ mesh: Mesh('fsdp': 1, 'tp': 1, axis_types=(Auto, Auto))
✅ tokenizer loaded
Loading params (this can take a bit)...
✅ models ready: True True


In [2]:
# ============================================================
# Cell B) Sampler + generate_one() + baseline outputs (CPU-safe)
# ============================================================
import re
import jax
from tunix.generate import sampler as sampler_lib
from tunix.models.gemma import model as gemma_lib

# ---- CPU-safe limits (keeps it from feeling like a hang) ----
BACKEND = jax.default_backend()
PROMPT_LEN = 128
GEN_TOKENS = 64
CACHE_SIZE = PROMPT_LEN + GEN_TOKENS + 256

model_config = gemma_lib.ModelConfig.gemma2_2b()

sampler = sampler_lib.Sampler(
    transformer=actor_model,
    tokenizer=tokenizer,
    cache_config=sampler_lib.CacheConfig(
        cache_size=int(CACHE_SIZE),
        num_layers=int(model_config.num_layers),
        num_kv_heads=int(model_config.num_kv_heads),
        head_dim=int(model_config.head_dim),
    ),
)

print("✅ Sampler ready | backend:", BACKEND, "| PROMPT_LEN:", PROMPT_LEN, "| GEN_TOKENS:", GEN_TOKENS)

SYSTEM_PROMPT = (
    "You are a helpful assistant.\n"
    "Always respond in this exact format:\n"
    "<reasoning>...</reasoning>\n"
    "<answer>...</answer>\n"
    "Keep the <answer> concise."
)

TEMPLATE = (
    "<start_of_turn>system\n{system_prompt}\n<end_of_turn>\n"
    "<start_of_turn>user\n{question}\n<end_of_turn>\n"
)

def generate_one(question: str, temperature=0.7, top_k=50, top_p=0.95, seed=0) -> str:
    prompt = TEMPLATE.format(system_prompt=SYSTEM_PROMPT, question=question)
    out = sampler(
        input_strings=[prompt],
        max_generation_steps=int(GEN_TOKENS),
        temperature=float(temperature),
        top_k=int(top_k),
        top_p=float(top_p),
        echo=False,
        seed=int(seed),
    )
    return out.text[0]

# ---- Baseline outputs ----
print(generate_one("If Tom has 3 apples and buys 4 more, how many apples does he have?", temperature=0.7, seed=0))
print()
print(generate_one("Explain what is GRPO in simple terms.", temperature=0.7, seed=1))


✅ Sampler ready | backend: cpu | PROMPT_LEN: 128 | GEN_TOKENS: 64
<reasoning> To find the total number of apples, we need to add the number he already has to the number he buys.  <answer>7</answer> 
<end_of_turn>

<reasoning>GRPO stands for "Group Research Project Organizers."  It's a type of project management framework that's used for collaborative research projects.  It helps teams to organize and manage their research efforts, from planning and execution to communication and evaluation. </reasoning>
<answer>GR


In [5]:
# ---- Create a minimal CFG if missing, then print it ----
import os
import pprint
from types import SimpleNamespace

if "CFG" not in globals():
    CFG = SimpleNamespace(
        # paths
        ckpt_dir="/kaggle/working/ckpts",
        model_kagglehub_id="google/gemma-2/flax/gemma2-2b-it/1",
        params_subdir="gemma2-2b-it",

        # sampler / rollout
        max_prompt_len=128,
        max_gen_tokens=64,
        temperature=0.7,
        top_k=50,
        top_p=0.95,

        # GRPO
        lr=2e-6,
        beta=0.08,
        epsilon=0.2,
        num_generations=4,   # must be > 1 for GRPO
        num_iterations=1,

        # training loop sizes
        max_steps=200,
        eval_every=50,
        save_every=50,
        max_to_keep=2,

        # batching
        mini_bs=1,
        train_micro_bs=1,
        rollout_micro_bs=1,
    )
    print("✅ CFG created (minimal defaults).")

print("\n--- CFG values ---")
pprint.pprint(vars(CFG), sort_dicts=True, width=120)



--- CFG values ---
{'beta': 0.08,
 'ckpt_dir': '/kaggle/working/ckpts',
 'epsilon': 0.2,
 'eval_every': 50,
 'lr': 2e-06,
 'max_gen_tokens': 64,
 'max_prompt_len': 128,
 'max_steps': 200,
 'max_to_keep': 2,
 'mini_bs': 1,
 'model_kagglehub_id': 'google/gemma-2/flax/gemma2-2b-it/1',
 'num_generations': 4,
 'num_iterations': 1,
 'params_subdir': 'gemma2-2b-it',
 'rollout_micro_bs': 1,
 'save_every': 50,
 'temperature': 0.7,
 'top_k': 50,
 'top_p': 0.95,
 'train_micro_bs': 1}


In [9]:
need = ["CFG", "train_gsm8k", "test_gsm8k", "extra_ds", "grpo_trainer"]
missing = [x for x in need if x not in globals()]
print("Missing:", missing)
print("Backend:", jax.default_backend(), "| Devices:", jax.devices())


Missing: ['train_gsm8k', 'test_gsm8k', 'extra_ds', 'grpo_trainer']
Backend: cpu | Devices: [CpuDevice(id=0)]


Cell A → builds train_gsm8k, test_gsm8k, extra_ds

In [12]:
# ============================================================
# Cell A) Datasets: train_gsm8k / test_gsm8k / extra_ds
# Self-contained (creates TEMPLATE/SYSTEM_PROMPT if missing)
# ============================================================
import re
import grain
import tensorflow_datasets as tfds
import tensorflow_datasets.text.gsm8k  # registers gsm8k

# --- Ensure CFG exists (minimal) ---
from types import SimpleNamespace
if "CFG" not in globals():
    CFG = SimpleNamespace(
        train_micro_bs=1,
        max_prompt_len=128,
        max_gen_tokens=64,
    )
    print("✅ CFG created (minimal)")

# --- Ensure prompt template exists ---
if "SYSTEM_PROMPT" not in globals():
    SYSTEM_PROMPT = (
        "You are a helpful assistant.\n"
        "Always respond in this exact format:\n"
        "<reasoning>...</reasoning>\n"
        "<answer>...</answer>\n"
        "Keep the <answer> concise."
    )

if "TEMPLATE" not in globals():
    TEMPLATE = (
        "<start_of_turn>system\n{system_prompt}\n<end_of_turn>\n"
        "<start_of_turn>user\n{question}\n<end_of_turn>\n"
    )

def extract_hash_answer(text: str):
    # GSM8K answers are like "... #### 13"
    if "####" not in text:
        return None
    return text.split("####", 1)[1].strip()

def build_gsm8k(split: str, batch_size: int, data_dir: str):
    ds = tfds.data_source(
        "gsm8k",
        split=split,
        data_dir=data_dir,
        builder_kwargs={"file_format": tfds.core.FileFormat.ARRAY_RECORD},
        download=True,
    )

    def _as_text(v):
        return v if isinstance(v, str) else v.decode("utf-8")

    return (
        grain.MapDataset.source(ds)
        .shuffle(seed=42)
        .map(lambda x: {
            "prompts": TEMPLATE.format(system_prompt=SYSTEM_PROMPT, question=_as_text(x["question"])),
            "question": _as_text(x["question"]),
            "answer": extract_hash_answer(_as_text(x["answer"])),
        })
        .batch(batch_size)
    )

EXTRA_TASKS = [
    ("Summarize in 2 sentences: A transformer uses attention to model relationships in text.", None),
    ("Write Python code for Fibonacci(n). Mention time complexity.", None),
    ("Explain why the sky is blue in simple terms.", None),
    ("Give 5 creative product ideas for a chess-themed birthday party.", None),
    ("Explain what overfitting is and how to reduce it.", None),
]

def build_extra(batch_size: int):
    rows = [{
        "prompts": TEMPLATE.format(system_prompt=SYSTEM_PROMPT, question=q),
        "question": q,
        "answer": a
    } for q, a in EXTRA_TASKS]
    return grain.MapDataset.source(rows).shuffle(seed=7).batch(batch_size)

TRAIN_DIR = "/kaggle/working/data/train"
TEST_DIR  = "/kaggle/working/data/test"

BS = int(getattr(CFG, "train_micro_bs", 1))
train_gsm8k = build_gsm8k("train", BS, TRAIN_DIR)
test_gsm8k  = build_gsm8k("test",  BS, TEST_DIR)
extra_ds    = build_extra(BS)

print("✅ Datasets ready:",
      "train_gsm8k" in globals(),
      "test_gsm8k" in globals(),
      "extra_ds" in globals())

# quick peek
b = next(iter(train_gsm8k[:1]))
print("Batch keys:", b.keys())
print("Example Q:", b["question"][0][:80], "...")
print("Gold A:", b["answer"][0])


✅ Datasets ready: True True True
Batch keys: dict_keys(['answer', 'prompts', 'question'])
Example Q: Jane is painting her fingernails. She applies a base coat that takes 2 minutes t ...
Gold A: 13


Cell B → builds grpo_trainer (and a generate_one helper) and runs a CPU-safe SMOKE train (won’t “hang”)

In [14]:
import os

# Hard-disable CUDA plugin discovery (prevents xla_cuda12 initialize crash)
os.environ["JAX_PLATFORMS"] = "cpu"
os.environ["JAX_DISABLE_MOST_PJRT"] = "1"
os.environ["JAX_TRACEBACK_FILTERING"] = "off"


In [ ]:
import os, re, gc, inspect
import jax
import optax
import kagglehub

from types import SimpleNamespace
from tunix.generate import sampler as sampler_lib
from tunix.generate import tokenizer_adapter as tokenizer_lib
from tunix.models.gemma import model as gemma_lib
from tunix.models.gemma import params as params_lib
from tunix.rl import rl_cluster as rl_cluster_lib
from tunix.rl.rollout import base_rollout
from tunix.rl.grpo.grpo_learner import GRPOConfig, GRPOLearner

print("Backend:", jax.default_backend(), "| Devices:", jax.devices())

# ---- CFG minimal (CPU smoke) ----
CFG = SimpleNamespace(
    ckpt_dir="/kaggle/working/ckpts",
    model_kagglehub_id="google/gemma-2/flax/gemma2-2b-it/1",
    params_subdir="gemma2-2b-it",

    max_prompt_len=64,
    max_gen_tokens=16,
    temperature=0.7,
    top_k=50,
    top_p=0.95,

    lr=2e-6,
    beta=0.08,
    epsilon=0.2,
    num_generations=2,   # MUST be > 1
    num_iterations=1,

    max_steps=1,
    eval_every=1,

    mini_bs=1,
    train_micro_bs=1,
    rollout_micro_bs=1,
)

os.makedirs(CFG.ckpt_dir, exist_ok=True)

# ---- Download checkpoint ----
CKPT_ROOT = kagglehub.model_download(CFG.model_kagglehub_id)
TOKENIZER_PATH = os.path.join(CKPT_ROOT, "tokenizer.model")
PARAMS_DIR = os.path.join(CKPT_ROOT, CFG.params_subdir)

assert os.path.exists(TOKENIZER_PATH)
assert os.path.exists(PARAMS_DIR)

# ---- Mesh ----
mesh = jax.make_mesh((1, 1), ("fsdp", "tp"), axis_types=(jax.sharding.AxisType.Auto, jax.sharding.AxisType.Auto))
print("mesh:", mesh)

# ---- Tokenizer ----
tokenizer = tokenizer_lib.Tokenizer(tokenizer_path=TOKENIZER_PATH)

# ---- Build ONE model on CPU (reuse for ref + actor to save RAM) ----
print("Loading params (CPU)…")
params = params_lib.load_and_format_params(PARAMS_DIR)
model = gemma_lib.Transformer.from_params(params, version="2-2b-it")
del params
gc.collect()

actor_model = model
ref_model = model
print("✅ model built (shared actor/ref)")

# ---- Sampler + generate_one ----
model_config = gemma_lib.ModelConfig.gemma2_2b()
CACHE_SIZE = int(CFG.max_prompt_len + CFG.max_gen_tokens + 256)

sampler = sampler_lib.Sampler(
    transformer=actor_model,
    tokenizer=tokenizer,
    cache_config=sampler_lib.CacheConfig(
        cache_size=CACHE_SIZE,
        num_layers=int(model_config.num_layers),
        num_kv_heads=int(model_config.num_kv_heads),
        head_dim=int(model_config.head_dim),
    ),
)

SYSTEM_PROMPT = "You are a helpful math reasoning assistant."
TEMPLATE = "{system_prompt}\n\nQuestion: {question}\n\n"

def generate_one(question: str, seed: int = 0) -> str:
    prompt = TEMPLATE.format(system_prompt=SYSTEM_PROMPT, question=question)
    out = sampler(
        input_strings=[prompt],
        max_generation_steps=int(CFG.max_gen_tokens),
        temperature=float(CFG.temperature),
        top_k=int(CFG.top_k),
        top_p=float(CFG.top_p),
        echo=False,
        seed=int(seed),
    )
    return out.text[0]

# ---- Reward fns (simple + safe) ----
REASONING_START, REASONING_END = "<reasoning>", "</reasoning>"
ANSWER_START, ANSWER_END = "<answer>", "</answer>"

FORMAT_RE = re.compile(rf"{re.escape(REASONING_START)}.*?{re.escape(REASONING_END)}.*?{re.escape(ANSWER_START)}.*?{re.escape(ANSWER_END)}", re.DOTALL)
NUM_RE = re.compile(rf"{re.escape(ANSWER_START)}\s*([-+]?\d[\d,]*\.?\d*)\s*{re.escape(ANSWER_END)}", re.DOTALL)

def r_format(prompts, completions, **kwargs):
    return [1.0 if FORMAT_RE.search(c) else -0.5 for c in completions]

def r_math(prompts, completions, answer=None, **kwargs):
    if answer is None:
        return [0.0 for _ in completions]
    out = []
    for c, a in zip(completions, answer):
        m = NUM_RE.search(c)
        if not m:
            out.append(-0.2); continue
        try:
            pred = float(m.group(1).replace(",", ""))
            gold = float(str(a).replace(",", "").strip())
            out.append(1.0 if abs(pred-gold) < 1e-9 else -0.5)
        except:
            out.append(-0.3)
    return out

reward_fns = [r_format, r_math]

# ---- Optimizer ----
optimizer = optax.adamw(learning_rate=float(CFG.lr))

EOS_TOKENS = [1]

training_cfg = rl_cluster_lib.RLTrainingConfig(
    actor_optimizer=optimizer,
    eval_every_n_steps=int(CFG.eval_every),
    max_steps=int(CFG.max_steps),
    mini_batch_size=int(CFG.mini_bs),
    train_micro_batch_size=int(CFG.train_micro_bs),
    rollout_micro_batch_size=int(CFG.rollout_micro_bs),
    checkpoint_root_directory=str(CFG.ckpt_dir),
)

rollout_cfg = base_rollout.RolloutConfig(
    max_tokens_to_generate=int(CFG.max_gen_tokens),
    max_prompt_length=int(CFG.max_prompt_len),
    kv_cache_size=int(CFG.max_prompt_len + CFG.max_gen_tokens + 256),
    temperature=float(CFG.temperature),
    top_p=float(CFG.top_p),
    top_k=int(CFG.top_k),
    eos_tokens=EOS_TOKENS,
)

cluster_cfg = rl_cluster_lib.ClusterConfig(
    role_to_mesh={
        rl_cluster_lib.Role.ACTOR: mesh,
        rl_cluster_lib.Role.REFERENCE: mesh,
        rl_cluster_lib.Role.ROLLOUT: mesh,
    },
    rollout_engine="vanilla",
    offload_to_cpu=False,
    training_config=training_cfg,
    rollout_config=rollout_cfg,
)

rl_cluster = rl_cluster_lib.RLCluster(
    actor=actor_model,
    reference=ref_model,
    tokenizer=tokenizer,
    cluster_config=cluster_cfg,
)

grpo_cfg = GRPOConfig(
    num_generations=int(CFG.num_generations),
    num_iterations=int(CFG.num_iterations),
    beta=float(CFG.beta),
    epsilon=float(CFG.epsilon),
)

# Handle API differences
sig = inspect.signature(GRPOLearner)
kwargs = {"rl_cluster": rl_cluster, "reward_fns": reward_fns}
if "grpo_config" in sig.parameters:
    kwargs["grpo_config"] = grpo_cfg
elif "algo_config" in sig.parameters:
    kwargs["algo_config"] = grpo_cfg

grpo_trainer = GRPOLearner(**kwargs)

print("✅ grpo_trainer ready (NO TRAINING RUN HERE)")
print("✅ generate_one test:")
print(generate_one("2+2=?", seed=0)[:200])


Backend: cpu | Devices: [CpuDevice(id=0)]


mesh: Mesh('fsdp': 1, 'tp': 1, axis_types=(Auto, Auto))
Loading params (CPU)…


✅ model built (shared actor/ref)


✅ grpo_trainer ready (NO TRAINING RUN HERE)
✅ generate_one test:


## 11) Train (guarded, so notebook won’t crash if API name differs)
Tunix versions sometimes rename the training entrypoint. This cell tries common names.
If none found, it prints what methods exist.

In [1]:
# ============================================================
# REBUILD CELL: CFG + model + sampler + reward fns + GRPO trainer
#              + minimal datasets (gsm8k + extra)
# CPU-safe, avoids long runs
# ============================================================

import os, re, gc, inspect
from types import SimpleNamespace
import kagglehub
import jax
import optax

from tunix.generate import sampler as sampler_lib
from tunix.generate import tokenizer_adapter as tokenizer_lib
from tunix.models.gemma import model as gemma_lib
from tunix.models.gemma import params as params_lib
from tunix.rl import rl_cluster as rl_cluster_lib
from tunix.rl.rollout import base_rollout
from tunix.rl.grpo.grpo_learner import GRPOConfig, GRPOLearner

print("Backend:", jax.default_backend(), "| Devices:", jax.devices())

# ---- CFG (CPU-safe defaults) ----
CFG = SimpleNamespace(
    ckpt_dir="/kaggle/working/ckpts",
    model_kagglehub_id="google/gemma-2/flax/gemma2-2b-it/1",
    params_subdir="gemma2-2b-it",

    max_prompt_len=64,
    max_gen_tokens=16,
    temperature=0.7,
    top_k=50,
    top_p=0.95,

    lr=2e-6,
    beta=0.08,
    epsilon=0.2,
    num_generations=2,   # MUST be >=2 for GRPO
    num_iterations=1,

    max_steps=1,         # smoke
    eval_every=1,

    mini_bs=1,
    train_micro_bs=1,
    rollout_micro_bs=1,
)
os.makedirs(CFG.ckpt_dir, exist_ok=True)
print("✅ CFG ready")

# ---- Download/checkpoint paths ----
CKPT_ROOT = kagglehub.model_download(CFG.model_kagglehub_id)
TOKENIZER_PATH = os.path.join(CKPT_ROOT, "tokenizer.model")
PARAMS_DIR = os.path.join(CKPT_ROOT, CFG.params_subdir)
assert os.path.exists(TOKENIZER_PATH), TOKENIZER_PATH
assert os.path.exists(PARAMS_DIR), PARAMS_DIR
print("✅ CKPT_ROOT:", CKPT_ROOT)

# ---- Mesh ----
mesh = jax.make_mesh((1, 1), ("fsdp", "tp"), axis_types=(jax.sharding.AxisType.Auto, jax.sharding.AxisType.Auto))
print("✅ mesh:", mesh)

# ---- Tokenizer ----
tokenizer = tokenizer_lib.Tokenizer(tokenizer_path=TOKENIZER_PATH)
print("✅ tokenizer loaded")

# ---- Build ONE model on CPU (share actor/ref to save RAM) ----
print("Loading params + building model…")
params = params_lib.load_and_format_params(PARAMS_DIR)
model = gemma_lib.Transformer.from_params(params, version="2-2b-it")
del params
gc.collect()

actor_model = model
ref_model = model
print("✅ model built (shared actor/ref)")

# ---- Sampler + generate_one ----
model_config = gemma_lib.ModelConfig.gemma2_2b()
CACHE_SIZE = int(CFG.max_prompt_len + CFG.max_gen_tokens + 256)

sampler = sampler_lib.Sampler(
    transformer=actor_model,
    tokenizer=tokenizer,
    cache_config=sampler_lib.CacheConfig(
        cache_size=CACHE_SIZE,
        num_layers=int(model_config.num_layers),
        num_kv_heads=int(model_config.num_kv_heads),
        head_dim=int(model_config.head_dim),
    ),
)

SYSTEM_PROMPT = "Return outputs in <reasoning>...</reasoning><answer>...</answer>."
TEMPLATE = "{system_prompt}\n\nQuestion: {question}\n\n"

def generate_one(question: str, seed: int = 0) -> str:
    prompt = TEMPLATE.format(system_prompt=SYSTEM_PROMPT, question=question)
    out = sampler(
        input_strings=[prompt],
        max_generation_steps=int(CFG.max_gen_tokens),
        temperature=float(CFG.temperature),
        top_k=int(CFG.top_k),
        top_p=float(CFG.top_p),
        echo=False,
        seed=int(seed),
    )
    return out.text[0]

print("✅ generate_one ready:", generate_one("2+2=?", seed=0)[:120])

# ---- Reward fns ----
REASONING_START, REASONING_END = "<reasoning>", "</reasoning>"
ANSWER_START, ANSWER_END = "<answer>", "</answer>"

FORMAT_RE = re.compile(rf"{re.escape(REASONING_START)}.*?{re.escape(REASONING_END)}.*?{re.escape(ANSWER_START)}.*?{re.escape(ANSWER_END)}", re.DOTALL)
NUM_RE = re.compile(rf"{re.escape(ANSWER_START)}\s*([-+]?\d[\d,]*\.?\d*)\s*{re.escape(ANSWER_END)}", re.DOTALL)

def r_format(prompts, completions, **kwargs):
    return [1.0 if FORMAT_RE.search(c) else -0.5 for c in completions]

def r_math(prompts, completions, answer=None, **kwargs):
    if answer is None:
        return [0.0 for _ in completions]
    out = []
    for c, a in zip(completions, answer):
        m = NUM_RE.search(c)
        if not m:
            out.append(-0.2); continue
        try:
            pred = float(m.group(1).replace(",", ""))
            gold = float(str(a).replace(",", "").strip())
            out.append(1.0 if abs(pred-gold) < 1e-9 else -0.5)
        except:
            out.append(-0.3)
    return out

reward_fns = [r_format, r_math]
print("✅ reward fns ready")

# ---- Optimizer ----
optimizer = optax.adamw(learning_rate=float(CFG.lr))
EOS_TOKENS = [1]

training_cfg = rl_cluster_lib.RLTrainingConfig(
    actor_optimizer=optimizer,
    eval_every_n_steps=int(CFG.eval_every),
    max_steps=int(CFG.max_steps),
    mini_batch_size=int(CFG.mini_bs),
    train_micro_batch_size=int(CFG.train_micro_bs),
    rollout_micro_batch_size=int(CFG.rollout_micro_bs),
    checkpoint_root_directory=str(CFG.ckpt_dir),
)

rollout_cfg = base_rollout.RolloutConfig(
    max_tokens_to_generate=int(CFG.max_gen_tokens),
    max_prompt_length=int(CFG.max_prompt_len),
    kv_cache_size=int(CFG.max_prompt_len + CFG.max_gen_tokens + 256),
    temperature=float(CFG.temperature),
    top_p=float(CFG.top_p),
    top_k=int(CFG.top_k),
    eos_tokens=EOS_TOKENS,
)

cluster_cfg = rl_cluster_lib.ClusterConfig(
    role_to_mesh={
        rl_cluster_lib.Role.ACTOR: mesh,
        rl_cluster_lib.Role.REFERENCE: mesh,
        rl_cluster_lib.Role.ROLLOUT: mesh,
    },
    rollout_engine="vanilla",
    offload_to_cpu=False,
    training_config=training_cfg,
    rollout_config=rollout_cfg,
)

rl_cluster = rl_cluster_lib.RLCluster(
    actor=actor_model,
    reference=ref_model,
    tokenizer=tokenizer,
    cluster_config=cluster_cfg,
)

grpo_cfg = GRPOConfig(
    num_generations=int(CFG.num_generations),
    num_iterations=int(CFG.num_iterations),
    beta=float(CFG.beta),
    epsilon=float(CFG.epsilon),
)

# API differences handled here
sig = inspect.signature(GRPOLearner)
kwargs = {"rl_cluster": rl_cluster, "reward_fns": reward_fns}
if "grpo_config" in sig.parameters:
    kwargs["grpo_config"] = grpo_cfg
elif "algo_config" in sig.parameters:
    kwargs["algo_config"] = grpo_cfg

grpo_trainer = GRPOLearner(**kwargs)
print("✅ grpo_trainer ready")

# ---- Minimal dummy datasets so train doesn't error ----
# If you already have real build_gsm8k/build_extra elsewhere, keep using those.
def build_gsm8k(split="train", batch_size=1):
    # Tiny synthetic: only for smoke (no download needed)
    data = [
        {"question": "2+2=?", "answer": "4"},
        {"question": "3+5=?", "answer": "8"},
    ]
    def gen():
        for ex in data:
            yield {"question": [ex["question"]], "answer": [ex["answer"]]}
    return gen()

def build_extra(batch_size=1):
    data = [{"question": "Return <reasoning>...</reasoning><answer>...</answer> exactly.", "answer": None}]
    def gen():
        for ex in data:
            yield {"question": [ex["question"]], "answer": [ex["answer"]]}
    return gen()

train_gsm8k = build_gsm8k("train", batch_size=1)
test_gsm8k  = build_gsm8k("test", batch_size=1)
extra_ds    = list(build_extra(batch_size=1))

print("✅ datasets ready: train_gsm8k/test_gsm8k/extra_ds")


/usr/local/lib/python3.12/dist-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'repr' attribute with value False was provided to the `Field()` function, which has no effect in the context it was used. 'repr' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` statement was used, or if the `Field()` function was attached to a single member of a union type.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'frozen' attribute with value True was provided to the `Field()` function, which has no effect in the context it was used. 'frozen' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` 

Backend: cpu | Devices: [CpuDevice(id=0)]
✅ CFG ready


✅ CKPT_ROOT: /kaggle/input/gemma-2/flax/gemma2-2b-it/1
✅ mesh: Mesh('fsdp': 1, 'tp': 1, axis_types=(Auto, Auto))
✅ tokenizer loaded
Loading params + building model…
✅ model built (shared actor/ref)


✅ generate_one ready: <reasoning>To answer this question, we need to use basic arithmetic.
✅ reward fns ready


✅ grpo_trainer ready
✅ datasets ready: train_gsm8k/test_gsm8k/extra_ds


In [2]:
import jax

need = ["grpo_trainer", "CFG", "build_gsm8k", "build_extra"]
missing = [k for k in need if k not in globals()]
print("Missing:", missing)
print("Backend:", jax.default_backend(), "| Devices:", jax.devices())

# CPU-safe overrides so training doesn't look stuck
if "CFG" in globals() and jax.default_backend() == "cpu":
    CFG.max_prompt_len = 64
    CFG.max_gen_tokens = 16
    CFG.num_generations = max(2, int(getattr(CFG, "num_generations", 2)))  # must be >=2
    CFG.max_steps = min(int(getattr(CFG, "max_steps", 1)), 2)
    print("✅ CPU-safe CFG overrides applied:")
    print("  max_steps:", CFG.max_steps,
          "| num_generations:", CFG.num_generations,
          "| prompt_len:", CFG.max_prompt_len,
          "| gen_tokens:", CFG.max_gen_tokens)


Missing: []
Backend: cpu | Devices: [CpuDevice(id=0)]
✅ CPU-safe CFG overrides applied:
  max_steps: 1 | num_generations: 2 | prompt_len: 64 | gen_tokens: 16


In [3]:
# Quick dataset sanity
print("train_gsm8k type:", type(train_gsm8k))
try:
    it = iter(train_gsm8k)
    b = next(it)
    print("✅ First train batch keys:", b.keys())
    print("✅ First train batch lens:", {k: (len(v) if hasattr(v, "__len__") else type(v)) for k,v in b.items()})
except StopIteration:
    print("❌ train_gsm8k is EMPTY (StopIteration)")
except Exception as e:
    print("❌ train_gsm8k iteration error:", repr(e))


train_gsm8k type: <class 'generator'>
✅ First train batch keys: dict_keys(['question', 'answer'])
✅ First train batch lens: {'question': 1, 'answer': 1}


In [4]:
import os

# Must be set BEFORE importing jax
os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"] = "false"
os.environ["XLA_PYTHON_CLIENT_MEM_FRACTION"] = "0.60"   # lower = safer on T4
os.environ["JAX_PLATFORMS"] = "cuda"                    # force GPU (fail if not available)

print("✅ Env set. NOW restart the kernel, then run next cell.")



✅ Env set. NOW restart the kernel, then run next cell.


In [5]:
!pip -q install -U "jax[cuda12]" -f https://storage.googleapis.com/jax-releases/jax_cuda_releases.html


/usr/lib/python3.12/pty.py:95: RuntimeWarning: os.fork() was called. os.fork() is incompatible with multithreaded code, and JAX is multithreaded, so this will likely lead to a deadlock.
  pid, fd = os.forkpty()


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.7/5.7 MB 47.9 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 153.7/153.7 MB 12.2 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.5/40.5 MB 49.9 MB/s eta 0:00:00:00:0100:01
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
cudf-cu12 25.6.0 requires pyarrow<20.0.0a0,>=14.0.0; platform_machine == "x86_64", but you have pyarrow 22.0.0 which is incompatible.


In [6]:
import jax
print("Backend:", jax.default_backend())
print("Devices:", jax.devices())


Backend: cpu
Devices: [CpuDevice(id=0)]


In [7]:
import os
os.environ["JAX_PLATFORMS"] = "cpu"
os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"] = "false"
print("✅ Forced CPU. Restart kernel now.")


✅ Forced CPU. Restart kernel now.


In [9]:
import itertools
import numpy as np

def _to_str(x):
    """Convert anything to a safe string (never returns non-str)."""
    if x is None:
        return ""
    # bytes -> decode
    if isinstance(x, (bytes, bytearray)):
        try:
            return x.decode("utf-8", errors="ignore")
        except:
            return str(x)
    # numpy/jax scalars
    try:
        if hasattr(x, "dtype") and hasattr(x, "shape") and x.shape == ():
            return str(x.item())
    except:
        pass
    # dict/list -> stringify
    if isinstance(x, (dict, list, tuple)):
        return str(x)
    # normal
    return str(x)

def adapt_batch_strict(batch):
    """Always returns {'prompts': list[str], 'answer': list[str] (optional)}"""
    if not isinstance(batch, dict):
        batch = {"prompt": batch}

    # find a prompt field
    for k in ["prompts", "prompt", "question", "input", "query", "text", "problem"]:
        if k in batch:
            prompts = batch[k]
            break
    else:
        raise KeyError(f"No prompt-like key in batch keys={list(batch.keys())}")

    # normalize to list
    if isinstance(prompts, (str, bytes, bytearray)):
        prompts = [prompts]
    elif not isinstance(prompts, (list, tuple)):
        prompts = [prompts]

    prompts = [_to_str(p) for p in prompts]

    # drop empty prompts (optional, but helps)
    prompts = [p for p in prompts if isinstance(p, str)]
    if len(prompts) == 0:
        # return a dummy prompt to avoid crashing
        prompts = [""]

    out = {"prompts": prompts}

    # optional answers
    ans = None
    for k in ["answer", "answers", "label", "gold", "target"]:
        if k in batch:
            ans = batch[k]
            break
    if ans is not None:
        if isinstance(ans, (str, bytes, bytearray)):
            ans = [ans]
        elif not isinstance(ans, (list, tuple)):
            ans = [ans]
        out["answer"] = [_to_str(a) for a in ans]

    return out

def adapt_strict(ds):
    for b in ds:
        yield adapt_batch_strict(b)

# ---- DEBUG: inspect first 3 items AFTER strict adapt ----
def peek(ds, n=3, name="ds"):
    it = iter(ds)
    for i in range(n):
        try:
            b = next(it)
        except StopIteration:
            print(f"❌ {name} is empty at i={i}")
            return
        print(f"\n[{name} batch {i}] keys:", list(b.keys()))
        print(" type(prompts):", type(b["prompts"]), "len:", len(b["prompts"]))
        for j, p in enumerate(b["prompts"][:3]):
            print("  prompt", j, "type:", type(p), "sample:", repr(p[:120]))

# run peeks
print("🔎 Peeking adapted train_gsm8k...")
peek(adapt_strict(train_gsm8k), n=2, name="train")
print("\n🔎 Peeking adapted test_gsm8k...")
peek(adapt_strict(test_gsm8k), n=1, name="eval")
print("\n🔎 Peeking adapted extra_ds...")
peek(adapt_strict(extra_ds), n=1, name="extra")

print("\n✅ If all prompt types above are <class 'str'> you are safe to train.")


🔎 Peeking adapted train_gsm8k...
❌ train is empty at i=0

🔎 Peeking adapted test_gsm8k...

[eval batch 0] keys: ['prompts', 'answer']
 type(prompts): <class 'list'> len: 1
  prompt 0 type: <class 'str'> sample: '3+5=?'

🔎 Peeking adapted extra_ds...

[extra batch 0] keys: ['prompts', 'answer']
 type(prompts): <class 'list'> len: 1
  prompt 0 type: <class 'str'> sample: 'Return <reasoning>...</reasoning><answer>...</answer> exactly.'

✅ If all prompt types above are <class 'str'> you are safe to train.


In [11]:
print("FIRST train batch keys:", list(first_train.keys()) if isinstance(first_train, dict) else type(first_train))
print("FIRST train batch sample:", first_train)


NameError: name 'first_train' is not defined

In [12]:
# ============================================================
# Tokenizer Safety Patch (hard-fix: SentencePiece "not a string")
# Run ONCE after tokenizer is created and before training.
# ============================================================

import numpy as np

# 1) Patch the Tunix tokenizer adapter to always coerce to Python str
_orig_encode = tokenizer.encode

def _safe_encode(x, **kwargs):
    # Convert numpy/jax scalars / objects / bytes / None -> safe str
    if x is None:
        x = ""
    if isinstance(x, (bytes, bytearray)):
        try:
            x = x.decode("utf-8", errors="ignore")
        except Exception:
            x = str(x)
    # numpy scalar -> python scalar
    try:
        if isinstance(x, np.generic):
            x = x.item()
    except Exception:
        pass
    # final guarantee
    if not isinstance(x, str):
        x = str(x)
    return _orig_encode(x, **kwargs)

tokenizer.encode = _safe_encode
print("✅ Patched tokenizer.encode to always accept any input type.")


✅ Patched tokenizer.encode to always accept any input type.


In [13]:
FORMAT_RE = re.compile(
    rf"^\s*{re.escape(REASONING_START)}.+?{re.escape(REASONING_END)}.*?"
    rf"{re.escape(ANSWER_START)}.+?{re.escape(ANSWER_END)}\s*(<end_of_turn>)?\s*$",
    flags=re.DOTALL,
)


In [15]:
# ============================================================
# Cell 11.1) FIX RecursionError (tokenizer.encode patched twice)
# Run this ONCE, right before calling grpo_trainer.train(...)
# ============================================================

import numpy as np

# 1) Get the real SentencePiece processor underneath Tunix Tokenizer
sp = getattr(tokenizer, "_tokenizer", None)
if sp is None:
    raise AttributeError("❌ tokenizer._tokenizer not found. Cannot repair encode safely.")

# 2) Build a SAFE base encode that bypasses tokenizer.encode entirely
def _base_encode_sp(text: str):
    # SentencePiece wants python str
    return sp.EncodeAsIds(text)

# 3) Create a single safe wrapper (NO recursion possible)
def _safe_encode_once(x, **kwargs):
    if x is None:
        x = ""
    if isinstance(x, (bytes, bytearray)):
        try:
            x = x.decode("utf-8", errors="ignore")
        except Exception:
            x = str(x)
    try:
        if isinstance(x, np.generic):
            x = x.item()
    except Exception:
        pass
    if not isinstance(x, str):
        x = str(x)

    # bypass tokenizer.encode chain completely
    return _base_encode_sp(x)

# 4) Install patch (idempotent-ish): overwrite any previous patch
tokenizer.encode = _safe_encode_once

print("✅ tokenizer.encode repaired (no recursion). Quick test:", tokenizer.encode("2+2=?")[:10])


✅ tokenizer.encode repaired (no recursion). Quick test: [2, 235284, 235340, 235284, 61395, 1]


In [2]:
# ============================================================
# CELL 11 (CPU-SAFE, NO-RESTART)
# - Loads missing: tokenizer/mesh/actor_model/ref_model
# - Fixes tokenizer.encode recursion + forces strings
# - Rebuilds GSM8K datasets and safe infinite streams
# - Initializes RLCluster safely
# - Smoke-test: generate 1 short completion (no training step)
# ============================================================

import os, time, gc, inspect
import numpy as np
import jax
import jax.numpy as jnp
import grain
import tensorflow_datasets as tfds
import tensorflow_datasets.text.gsm8k  # registers gsm8k
import optax
from tunix.rl.rollout import base_rollout
from tunix.rl import rl_cluster as rl_cluster_lib
from tunix.rl.rollout import base_rollout
from tunix.rl.grpo.grpo_learner import GRPOConfig, GRPOLearner

print("Backend:", jax.default_backend(), "| Devices:", jax.devices())

# ---------------------------
# 0) CFG (fallback) + CPU-safe overrides
# ---------------------------
if "CFG" not in globals():
    from types import SimpleNamespace
    CFG = SimpleNamespace()
    print("✅ CFG created (fallback).")

CFG.ckpt_dir          = getattr(CFG, "ckpt_dir", "/kaggle/working/ckpts")
CFG.lr                = float(getattr(CFG, "lr", 2e-6))
CFG.beta              = float(getattr(CFG, "beta", 0.08))
CFG.epsilon           = float(getattr(CFG, "epsilon", 0.2))
CFG.num_generations   = 1   # keep tiny for CPU safety
CFG.num_iterations    = 1
CFG.max_steps         = 1
CFG.eval_every        = 1
CFG.mini_bs           = 1
CFG.train_micro_bs    = 1
CFG.rollout_micro_bs  = 1
CFG.max_prompt_len    = 48
CFG.max_gen_tokens    = 16
CFG.temperature       = float(getattr(CFG, "temperature", 0.7))
CFG.top_p             = float(getattr(CFG, "top_p", 0.95))
CFG.top_k             = int(getattr(CFG, "top_k", 20))
CFG.extra_every       = 50  # less frequent extras

os.makedirs(CFG.ckpt_dir, exist_ok=True)

print(
    "✅ CPU-safe overrides:",
    "max_steps=", CFG.max_steps,
    "| num_generations=", CFG.num_generations,
    "| prompt_len=", CFG.max_prompt_len,
    "| gen_tokens=", CFG.max_gen_tokens
)

# ---------------------------
# 1) Ensure required globals exist (tokenizer, mesh, actor_model, ref_model)
# ---------------------------
missing = [k for k in ["tokenizer", "mesh", "actor_model", "ref_model"] if k not in globals()]
if missing:
    print(f"⚠️ Missing globals: {missing} -> Auto-loading minimal Step-5 logic...")

    import kagglehub
    from tunix.generate import tokenizer_adapter as tokenizer_lib
    from tunix.models.gemma import model as gemma_lib
    from tunix.models.gemma import params as params_lib

    model_kagglehub_id = getattr(CFG, "model_kagglehub_id", "google/gemma-2/flax/gemma2-2b-it/1")
    params_subdir      = getattr(CFG, "params_subdir", "gemma2-2b-it")

    CKPT_ROOT = globals().get("CKPT_ROOT", None)
    if CKPT_ROOT is None:
        CKPT_ROOT = kagglehub.model_download(model_kagglehub_id)
    print("✅ CKPT_ROOT:", CKPT_ROOT)

    TOKENIZER_PATH = os.path.join(CKPT_ROOT, "tokenizer.model")
    PARAMS_DIR = os.path.join(CKPT_ROOT, params_subdir)

    # Mesh for CPU
    mesh = jax.make_mesh((1, 1), ("fsdp", "tp"),
                         axis_types=(jax.sharding.AxisType.Auto, jax.sharding.AxisType.Auto))
    print("✅ mesh:", mesh)

    tokenizer = tokenizer_lib.Tokenizer(tokenizer_path=TOKENIZER_PATH)
    print("✅ tokenizer loaded")

    # Build model on CPU
    cpu_dev = jax.devices()[0]
    print("Loading params + building model (CPU, bf16 cast)…")
    with jax.default_device(cpu_dev):
        params = params_lib.load_and_format_params(PARAMS_DIR)
        params = jax.tree_util.tree_map(
            lambda x: x.astype(jnp.bfloat16) if hasattr(x, "dtype") else x,
            params
        )
        model = gemma_lib.Transformer.from_params(params, version="2-2b-it")
        actor_model = model
        ref_model = model

    del params
    gc.collect()
    print("✅ actor_model/ref_model created (shared).")

# ---------------------------
# 2) Patch tokenizer.encode safely (no recursion + ensure str)
# ---------------------------
sp = getattr(tokenizer, "_tokenizer", None)
if sp is None:
    raise AttributeError("❌ tokenizer._tokenizer not found. Cannot repair encode safely.")

def _safe_encode_once(x, **kwargs):
    if x is None:
        x = ""
    if isinstance(x, (bytes, bytearray)):
        x = x.decode("utf-8", errors="ignore")
    try:
        if isinstance(x, np.generic):
            x = x.item()
    except Exception:
        pass
    if not isinstance(x, str):
        x = str(x)
    return sp.EncodeAsIds(x)

tokenizer.encode = _safe_encode_once
print("✅ Patched tokenizer.encode test:", tokenizer.encode("2+2=?")[:8])

# ---------------------------
# 3) Build GSM8K datasets (small batch, safe)
# ---------------------------
SYSTEM_PROMPT = (
    "You are a helpful assistant.\n"
    "Always respond in this exact format:\n"
    "<reasoning>...</reasoning>\n"
    "<answer>...</answer>\n"
    "Do not output anything outside the tags."
)

TEMPLATE = (
    "<start_of_turn>system\n{system_prompt}\n<end_of_turn>\n"
    "<start_of_turn>user\n{question}\n<end_of_turn>\n"
    "<start_of_turn>model\n"
)

def _as_text(v):
    if v is None:
        return ""
    if isinstance(v, str):
        return v
    if isinstance(v, (bytes, bytearray)):
        return v.decode("utf-8", errors="ignore")
    return str(v)

def extract_hash_answer(text: str):
    if not text or "####" not in text:
        return None
    return text.split("####", 1)[1].strip()

def build_gsm8k_dataset(split: str, batch_size: int, data_dir: str):
    ds = tfds.data_source(
        "gsm8k",
        split=split,
        data_dir=data_dir,
        builder_kwargs={"file_format": tfds.core.FileFormat.ARRAY_RECORD},
        download=True,
    )
    return (
        grain.MapDataset.source(ds)
        .shuffle(seed=42)
        .map(lambda x: {
            "prompts": TEMPLATE.format(
                system_prompt=SYSTEM_PROMPT,
                question=_as_text(x["question"]),
            ),
            "answer": extract_hash_answer(_as_text(x["answer"])),
        })
        .batch(batch_size)
    )

TRAIN_DIR = "/kaggle/working/data/train"
TEST_DIR  = "/kaggle/working/data/test"
os.makedirs(TRAIN_DIR, exist_ok=True)
os.makedirs(TEST_DIR, exist_ok=True)

BS = 1
train_gsm8k = build_gsm8k_dataset("train", BS, TRAIN_DIR)
test_gsm8k  = build_gsm8k_dataset("test",  BS, TEST_DIR)

print("✅ Datasets rebuilt.")

# ---------------------------
# 4) Streams: ensure prompts are list[str]
# ---------------------------
def ensure_str_list(xs):
    if isinstance(xs, (str, bytes, bytearray)):
        xs = [xs]
    if not isinstance(xs, (list, tuple)):
        xs = [xs]
    out = []
    for x in xs:
        if isinstance(x, (bytes, bytearray)):
            x = x.decode("utf-8", errors="ignore")
        if x is None:
            x = ""
        if not isinstance(x, str):
            x = str(x)
        out.append(x)
    return out

def adapt_batch(batch):
    prompts = batch.get("prompts", batch.get("prompt", None))
    if prompts is None:
        raise KeyError(f"Batch missing prompts key. keys={list(batch.keys())}")
    prompts = ensure_str_list(prompts)
    return {"prompts": prompts, "answer": batch.get("answer", [None]*len(prompts))}

def repeat_forever(ds):
    while True:
        for b in ds:
            yield b

def adapt_stream(ds, name="stream", peek_n=1):
    it = iter(ds)
    for i in range(peek_n):
        b = next(it)
        bb = adapt_batch(b)
        print(f"[{name} peek {i}] type={type(bb['prompts'][0])} | sample={repr(bb['prompts'][0][:120])}")
        yield bb
    for b in it:
        yield adapt_batch(b)

train_iter = adapt_stream(repeat_forever(train_gsm8k), name="train_stream", peek_n=1)
eval_iter  = adapt_stream(repeat_forever(test_gsm8k),  name="eval_stream",  peek_n=1)

print("✅ Streams ready.")

# ---------------------------
# 5) Init RLCluster (CPU-safe)
# ---------------------------
gc.collect()
try:
    jax.clear_caches()
except Exception:
    pass

# Tiny optimizer state (SGD)
optimizer = optax.sgd(learning_rate=float(CFG.lr))

training_cfg = rl_cluster_lib.RLTrainingConfig(
    actor_optimizer=optimizer,
    eval_every_n_steps=CFG.eval_every,
    max_steps=CFG.max_steps,
    mini_batch_size=CFG.mini_bs,
    train_micro_batch_size=CFG.train_micro_bs,
    rollout_micro_batch_size=CFG.rollout_micro_bs,
    checkpoint_root_directory=str(CFG.ckpt_dir),
)

# ---- SAFE rollout config for CPU (prevents cache overflow) ----
rollout_cfg = base_rollout.RolloutConfig(
    max_tokens_to_generate=int(CFG.max_gen_tokens),
    max_prompt_length=int(CFG.max_prompt_len),
    kv_cache_size=int(CFG.max_prompt_len + CFG.max_gen_tokens + 256),
    temperature=float(CFG.temperature),
    top_p=float(CFG.top_p),
    top_k=min(int(CFG.top_k), 20),
    eos_tokens=[1],
)


cluster_cfg = rl_cluster_lib.ClusterConfig(
    role_to_mesh={
        rl_cluster_lib.Role.ACTOR: mesh,
        rl_cluster_lib.Role.REFERENCE: mesh,
        rl_cluster_lib.Role.ROLLOUT: mesh,
    },
    rollout_engine="vanilla",
    offload_to_cpu=True,
    training_config=training_cfg,
    rollout_config=rollout_cfg,
)

print("✅ Configs built. Initializing RLCluster (CPU)…")
rl_cluster = rl_cluster_lib.RLCluster(
    actor=actor_model,
    reference=ref_model,
    tokenizer=tokenizer,
    cluster_config=cluster_cfg,
)
print("✅ RLCluster initialized.")

# ---------------------------
# 6) Smoke-test generation (NO TRAINING YET)
# ---------------------------
sample = next(iter(test_gsm8k))
prompt = sample["prompts"][0]
print("\n🧪 Smoke-test: encoding prompt tokens =", len(tokenizer.encode(prompt)))

print("✅ Cell 11 done (setup + RLCluster + smoke-test). Safe to go to Cell 12.")


/usr/local/lib/python3.12/dist-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'repr' attribute with value False was provided to the `Field()` function, which has no effect in the context it was used. 'repr' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` statement was used, or if the `Field()` function was attached to a single member of a union type.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'frozen' attribute with value True was provided to the `Field()` function, which has no effect in the context it was used. 'frozen' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` 

Backend: cpu | Devices: [CpuDevice(id=0)]
✅ CFG created (fallback).
✅ CPU-safe overrides: max_steps= 1 | num_generations= 1 | prompt_len= 48 | gen_tokens= 16
⚠️ Missing globals: ['tokenizer', 'mesh', 'actor_model', 'ref_model'] -> Auto-loading minimal Step-5 logic...


✅ CKPT_ROOT: /kaggle/input/gemma-2/flax/gemma2-2b-it/1
✅ mesh: Mesh('fsdp': 1, 'tp': 1, axis_types=(Auto, Auto))
✅ tokenizer loaded
Loading params + building model (CPU, bf16 cast)…
✅ actor_model/ref_model created (shared).
✅ Patched tokenizer.encode test: [2, 235284, 235340, 235284, 61395, 1]
✅ Datasets rebuilt.
✅ Streams ready.


✅ Configs built. Initializing RLCluster (CPU)…


✅ RLCluster initialized.

🧪 Smoke-test: encoding prompt tokens = 96
✅ Cell 11 done (setup + RLCluster + smoke-test). Safe to go to Cell 12.


## 12) Post-training eval (format rate + GSM8K exact rate on a small slice)

In [1]:
# ============================================================
# CELL 12A — Make sure test_gsm8k exists (rebuild only if missing)
# ============================================================
import os
import tensorflow_datasets as tfds
import tensorflow_datasets.text.gsm8k
import grain

# Fallback prompts (only used if you didn't keep them in globals)
SYSTEM_PROMPT = globals().get(
    "SYSTEM_PROMPT",
    "You are a helpful assistant.\n"
    "Always respond in this exact format:\n"
    "<reasoning>...</reasoning>\n"
    "<answer>...</answer>\n"
    "Do not output anything outside the tags."
)
TEMPLATE = globals().get(
    "TEMPLATE",
    "<start_of_turn>system\n{system_prompt}\n<end_of_turn>\n"
    "<start_of_turn>user\n{question}\n<end_of_turn>\n"
)

def _as_text(v):
    if v is None: return ""
    if isinstance(v, str): return v
    if isinstance(v, (bytes, bytearray)): return v.decode("utf-8", errors="ignore")
    return str(v)

def extract_hash_answer(text: str):
    if not text or "####" not in text:
        return None
    return text.split("####", 1)[1].strip()

def build_gsm8k_dataset(split: str, batch_size: int, data_dir: str):
    ds = tfds.data_source(
        "gsm8k",
        split=split,
        data_dir=data_dir,
        builder_kwargs={"file_format": tfds.core.FileFormat.ARRAY_RECORD},
        download=True,
    )
    return (
        grain.MapDataset.source(ds)
        .shuffle(seed=42)
        .map(lambda x: {
            "prompts": TEMPLATE.format(system_prompt=SYSTEM_PROMPT, question=_as_text(x["question"])),
            "answer": extract_hash_answer(_as_text(x["answer"])),
        })
        .batch(batch_size)
    )

if "test_gsm8k" not in globals():
    BS = int(getattr(CFG, "train_micro_bs", 1)) if "CFG" in globals() else 1
    TEST_DIR = "/kaggle/working/data/test"
    os.makedirs(TEST_DIR, exist_ok=True)
    test_gsm8k = build_gsm8k_dataset("test", BS, TEST_DIR)
    print("✅ Rebuilt test_gsm8k.")
else:
    print("✅ test_gsm8k already exists.")


✅ Rebuilt test_gsm8k.


In [3]:
missing = [k for k in ["rl_cluster", "CFG", "test_gsm8k"] if k not in globals()]
print("Missing:", missing)


Missing: []


In [6]:
# ============================================================
# CELL 12 — FAST sanity eval (NO generation, finishes quickly)
# What it checks:
# - test_gsm8k batches exist
# - prompts are strings
# - prompt token length vs CFG.max_prompt_len
# - expected <reasoning>/<answer> format regex (template present)
# ============================================================

import re, time
import numpy as np
import jax

print("Backend:", jax.default_backend(), "| Devices:", jax.devices())

need = ["CFG", "tokenizer", "test_gsm8k"]
missing = [k for k in need if k not in globals()]
if missing:
    raise NameError(f"Missing globals: {missing}. Re-run Cell 11 (and 12A that builds test_gsm8k).")

FORMAT_RE = re.compile(r"<reasoning>.*?</reasoning>\s*<answer>.*?</answer>", re.DOTALL | re.IGNORECASE)

sp = getattr(tokenizer, "_tokenizer", None)
if sp is None:
    raise AttributeError("tokenizer._tokenizer missing.")

MAX_PROMPT_TOKENS = int(getattr(CFG, "max_prompt_len", 48))

t0 = time.time()

# Take just 1 batch
b = next(iter(test_gsm8k))
prompts = b["prompts"]
answers = b["answer"]

# Ensure python strings
prompts = [p if isinstance(p, str) else str(p) for p in prompts]

token_lens = [len(sp.EncodeAsIds(p)) for p in prompts]
over = sum(1 for L in token_lens if L > MAX_PROMPT_TOKENS)

print("✅ batch_size:", len(prompts))
print("✅ prompt type:", type(prompts[0]))
print("✅ sample prompt head:\n", prompts[0][:300])
print("✅ token lengths:", token_lens[:10], ("..." if len(token_lens) > 10 else ""))
print(f"✅ prompts over CFG.max_prompt_len({MAX_PROMPT_TOKENS}):", over, "/", len(prompts))

# This just checks the regex itself (not model output)
print("✅ FORMAT_RE compiled ok:", bool(FORMAT_RE.pattern))

print("⏱️ sanity eval time:", round(time.time() - t0, 2), "sec")


Backend: cpu | Devices: [CpuDevice(id=0)]
✅ batch_size: 1
✅ prompt type: <class 'numpy.str_'>
✅ sample prompt head:
 <start_of_turn>system
You are a helpful assistant.
Always respond in this exact format:
<reasoning>...</reasoning>
<answer>...</answer>
Do not output anything outside the tags.
<end_of_turn>
<start_of_turn>user
Last year there were 50 students enrolled in a calligraphy class. This year, there was a 
✅ token lengths: [93] 
✅ prompts over CFG.max_prompt_len(48): 1 / 1
✅ FORMAT_RE compiled ok: True
⏱️ sanity eval time: 0.03 sec


In [4]:
# ============================================================
# CELL 12 — Post-training eval (CPU-safe, NO TRAINING, anti-restart)
# What this fixes:
# - Disables JIT best-effort (reduces compilation spikes)
# - Hard-truncates prompts to fit CFG.max_prompt_len tokens
# - Evaluates only a tiny slice first (prevents hard kernel restart)
# ============================================================

import re, time, gc
import jax

print("Backend:", jax.default_backend(), "| Devices:", jax.devices())

# ---------------------------
# 0) Must-have globals check
# ---------------------------
need = ["CFG", "tokenizer", "rl_cluster", "test_gsm8k"]
missing = [k for k in need if k not in globals()]
if missing:
    raise NameError(f"Missing globals: {missing}. Re-run Cell 11 (and 12A for test_gsm8k).")

# ---------------------------
# 1) Best-effort disable JIT (helps reduce CPU memory spikes)
# NOTE: This may not disable everything inside Tunix, but it often helps.
# ---------------------------
try:
    from jax import config as jax_config
    jax_config.update("jax_disable_jit", True)
    print("✅ jax_disable_jit=True (best-effort)")
except Exception as e:
    print("ℹ️ Could not set jax_disable_jit:", repr(e))

gc.collect()
try:
    jax.clear_caches()
except Exception:
    pass

# ---------------------------
# 2) Regex helpers
# ---------------------------
FORMAT_RE = re.compile(r"<reasoning>.*?</reasoning>\s*<answer>.*?</answer>", re.DOTALL | re.IGNORECASE)
NUM_RE    = re.compile(r"([-+]?\d+(?:\.\d+)?)")

def format_ok(text: str) -> bool:
    return bool(text) and (FORMAT_RE.search(text) is not None)

def extract_number(text: str):
    if not text:
        return None
    m = NUM_RE.search(text)
    return m.group(1).strip() if m else None

# ---------------------------
# 3) HARD token-truncation to prevent cache/memory blowups
# - keeps last max_prompt_len tokens (chat template is long!)
# ---------------------------
sp = getattr(tokenizer, "_tokenizer", None)
if sp is None:
    raise AttributeError("tokenizer._tokenizer missing; cannot token-truncate safely.")

MAX_PROMPT_TOKENS = int(getattr(CFG, "max_prompt_len", 48))
MAX_PROMPT_TOKENS = min(MAX_PROMPT_TOKENS, 48)  # keep tight on CPU

def truncate_prompt_by_tokens(prompt: str, max_tokens: int = MAX_PROMPT_TOKENS) -> str:
    if prompt is None:
        prompt = ""
    if not isinstance(prompt, str):
        prompt = str(prompt)

    ids = sp.EncodeAsIds(prompt)
    if len(ids) <= max_tokens:
        return prompt

    # Keep the most recent tokens (usually keeps the user question end)
    ids = ids[-max_tokens:]
    try:
        return sp.DecodeIds(ids)
    except Exception:
        # fallback: if decode not available for some reason
        return prompt[-2000:]  # last chars

# ---------------------------
# 4) Generation wrapper (super small, micro_batch=1)
# ---------------------------
def generate_one(prompt: str):
    prompt = truncate_prompt_by_tokens(prompt, MAX_PROMPT_TOKENS)

    try:
        out = rl_cluster.generate(
            prompts=[prompt],
            apply_chat_template=False,   # your prompts already include template
            mode="eval",
            micro_batch_size=1,
        )
    except Exception as e:
        # Important: don't crash eval loop
        return "", repr(e)

    # Normalize outputs
    txt = ""
    if isinstance(out, dict):
        if "completions" in out and out["completions"]:
            txt = out["completions"][0]
        elif "texts" in out and out["texts"]:
            txt = out["texts"][0]
    elif isinstance(out, (list, tuple)) and len(out) > 0:
        txt = out[0]

    return txt, None

# ---------------------------
# 5) Evaluate on a VERY small slice first
# (Increase slowly once stable)
# ---------------------------
def eval_gsm8k_small(ds, n_examples=5):
    total = fmt = exact = 0
    errors = 0

    for b in ds:
        prompts = b["prompts"]
        answers = b["answer"]

        for p, a in zip(prompts, answers):
            out, err = generate_one(p)
            if err is not None:
                errors += 1
                # continue without crashing
                continue

            total += 1
            if format_ok(out):
                fmt += 1

            g = extract_number(out)
            if g is not None and a is not None:
                try:
                    if float(g) == float(str(a).strip()):
                        exact += 1
                except Exception:
                    pass

            if total >= n_examples:
                return {
                    "total": total,
                    "format_rate": (fmt / total if total else 0.0),
                    "exact_rate": (exact / total if total else 0.0),
                    "errors": errors,
                    "max_prompt_tokens_used": MAX_PROMPT_TOKENS,
                }

    return {
        "total": total,
        "format_rate": (fmt / total if total else 0.0),
        "exact_rate": (exact / total if total else 0.0),
        "errors": errors,
        "max_prompt_tokens_used": MAX_PROMPT_TOKENS,
    }

t0 = time.time()
metrics = eval_gsm8k_small(test_gsm8k, n_examples=5)  # start tiny
print("📌 Post-training small-eval metrics:", metrics)
print("⏱️ Eval time:", round(time.time() - t0, 2), "sec")
metrics


Backend: cpu | Devices: [CpuDevice(id=0)]
✅ jax_disable_jit=True (best-effort)
📌 Post-training small-eval metrics: {'total': 5, 'format_rate': 0.0, 'exact_rate': 0.0, 'errors': 0, 'max_prompt_tokens_used': 48}
⏱️ Eval time: 547.84 sec


{'total': 5,
 'format_rate': 0.0,
 'exact_rate': 0.0,
 'errors': 0,
 'max_prompt_tokens_used': 48}

## 13) Cross-domain examples (judges love qualitative demos)

In [14]:
# DEBUG: show the raw output from rl_cluster.generate
q = "Summarize in 2 sentences: Reinforcement learning improves behavior using rewards."

prompt = TEMPLATE.format(system_prompt=SYSTEM_PROMPT, question=q)
prompt = truncate_prompt(prompt, MAX_PROMPT_TOKENS)

out = rl_cluster.generate(
    prompts=[prompt],
    apply_chat_template=False,
    mode="eval",
    micro_batch_size=1,
)

print("TYPE:", type(out))
print("OUT:", out)

# If it's a dict, show keys
if isinstance(out, dict):
    print("DICT KEYS:", list(out.keys()))


TYPE: <class 'tunix.rl.rollout.base_rollout.RolloutOutput'>
OUT: RolloutOutput(text=[''], logits=None, tokens=Array([[0, 0, 0, 0, 0, 0, 0, 0]], dtype=int32), left_padded_prompt_tokens=Array([[     0,      0,      0,      0,      0,      0,      0,      0,
             0,      0,      0,      0,      0,      0,      2,      2,
        235322,  21248,    574, 235313,  39744,  21248,    574, 235313,
           108, 235322,  13072, 235313,  39744,  13072, 235313,    108,
          2795,    780,   5033,   4341,   5162,    573,  16323, 235265,
           108,    107,    108,    106,   1645,    108, 156910,    889,
           575, 235248, 235284,  26099, 235292, 165808,   6044,  38291,
          8409,   2177,  28764, 235265,    108,    107,    108,      1]],      dtype=int32), logprobs=None)


In [3]:
# ============================================================
# CELL 13 — Cross-domain qualitative demos (CPU-safe, anti-restart, answers shown)
# ✅ Works with your Cell 11 globals:
#    CFG, tokenizer, rl_cluster, SYSTEM_PROMPT, TEMPLATE
# ✅ Avoids kernel restart:
#    - tiny prompt token budget
#    - tiny generation budget (CFG.max_gen_tokens, capped)
#    - runs only 2 examples by default
# ✅ Ensures Gemma chat format is correct:
#    - guarantees <start_of_turn>model marker
# ✅ Prints answers even if Tunix returns RolloutOutput
# ============================================================

import time, re, gc
import jax

print("Backend:", jax.default_backend(), "| Devices:", jax.devices())

# ---------------------------
# 0) Must-have globals
# ---------------------------
need = ["CFG", "tokenizer", "rl_cluster", "SYSTEM_PROMPT", "TEMPLATE"]
missing = [k for k in need if k not in globals()]
if missing:
    raise NameError(f"Missing globals: {missing}. Re-run Cell 11 first.")

sp = getattr(tokenizer, "_tokenizer", None)
if sp is None:
    raise AttributeError("tokenizer._tokenizer missing; cannot token-truncate safely.")

# ---------------------------
# 1) Safety caps (keep very small on CPU)
# ---------------------------
MAX_PROMPT_TOKENS = int(getattr(CFG, "max_prompt_len", 48))
MAX_PROMPT_TOKENS = min(MAX_PROMPT_TOKENS, 48)     # keep tight
MAX_NEW_TOKENS    = int(getattr(CFG, "max_gen_tokens", 16))
MAX_NEW_TOKENS    = min(max(8, MAX_NEW_TOKENS), 16)  # 8..16

# IMPORTANT: cache must be >= prompt+gen+buffer
try:
    rcfg = rl_cluster.cluster_config.rollout_config
    needed_cache = MAX_PROMPT_TOKENS + MAX_NEW_TOKENS + 64
    if getattr(rcfg, "kv_cache_size", 0) < needed_cache:
        print(f"⚠️ kv_cache_size={rcfg.kv_cache_size} too small; bumping to {needed_cache}")
        rcfg.kv_cache_size = int(needed_cache)
    rcfg.max_prompt_length = int(MAX_PROMPT_TOKENS)
    rcfg.max_tokens_to_generate = int(MAX_NEW_TOKENS)
except Exception as e:
    print("ℹ️ Could not patch rollout_config (ok):", repr(e))

# best-effort: reduce JIT memory spikes
try:
    from jax import config as jax_config
    jax_config.update("jax_disable_jit", True)
    print("✅ jax_disable_jit=True (best-effort)")
except Exception:
    pass

gc.collect()
try:
    jax.clear_caches()
except Exception:
    pass

# ---------------------------
# 2) Prompt builder + truncation (keeps system rules intact)
# ---------------------------
MODEL_MARKER = "<start_of_turn>model\n"

def make_prompt(question: str) -> str:
    q = "" if question is None else str(question)
    p = TEMPLATE.format(system_prompt=SYSTEM_PROMPT, question=q)
    # Ensure model turn exists (Gemma expects this)
    if MODEL_MARKER not in p:
        p = p + MODEL_MARKER
    return p

def truncate_prompt_keep_tail(prompt: str, max_tokens: int) -> str:
    # keep last tokens to prioritize the user question end + model marker
    ids = sp.EncodeAsIds(prompt)
    if len(ids) <= max_tokens:
        return prompt
    ids = ids[-max_tokens:]
    try:
        return sp.DecodeIds(ids)
    except Exception:
        return prompt[-2000:]

# ---------------------------
# 3) Output normalization (handles RolloutOutput, dict, list)
# ---------------------------
ANSWER_RE = re.compile(r"<answer>(.*?)</answer>", re.DOTALL | re.IGNORECASE)

def normalize_text(out):
    # RolloutOutput (common)
    if hasattr(out, "text"):
        try:
            return out.text[0] if out.text else ""
        except Exception:
            return ""
    # dict variants
    if isinstance(out, dict):
        for k in ["completions", "texts", "text"]:
            if k in out and out[k]:
                v = out[k]
                if isinstance(v, (list, tuple)):
                    return v[0] if v else ""
                return str(v)
        return ""
    # list/tuple
    if isinstance(out, (list, tuple)) and out:
        return out[0] if isinstance(out[0], str) else str(out[0])
    return ""

def extract_answer_only(text: str) -> str:
    if not text:
        return ""
    m = ANSWER_RE.search(text)
    return m.group(1).strip() if m else text.strip()

# ---------------------------
# 4) Generate one (tiny, micro_batch=1)
# ---------------------------
def generate_one_text(question: str):
    prompt = make_prompt(question)
    prompt = truncate_prompt_keep_tail(prompt, MAX_PROMPT_TOKENS)

    out = rl_cluster.generate(
        prompts=[prompt],
        apply_chat_template=False,   # prompt already has chat template
        mode="eval",
        micro_batch_size=1,
    )
    txt = normalize_text(out)
    return txt

# ---------------------------
# 5) Run qualitative demos (start with 2; increase if stable)
# ---------------------------
examples = [
    "Summarize in 2 sentences: Reinforcement learning improves behavior using rewards.",
    "Write a Python function to check palindrome. Mention complexity.",
    "Explain why ice floats on water.",
    "Give 5 creative ideas for a chess-themed school event.",
    "Explain what overfitting is and how to prevent it.",
]

RUN_N = 2  # ✅ safest on CPU. If stable, set 3 then 5.

print(f"✅ Using MAX_PROMPT_TOKENS={MAX_PROMPT_TOKENS} | MAX_NEW_TOKENS={MAX_NEW_TOKENS} | RUN_N={RUN_N}")

t0 = time.time()
for i, q in enumerate(examples[:RUN_N]):
    print("=" * 90)
    print("Q:", q)

    t1 = time.time()
    raw = generate_one_text(q)
    ans = extract_answer_only(raw)

    # ensure something prints
    if not ans:
        print("A: [EMPTY OUTPUT]  → Increase CFG.max_gen_tokens to 16 (already) and re-run Cell 11.")
        # helpful debug: show if model produced any tokens
        try:
            dbg = rl_cluster.generate(
                prompts=[truncate_prompt_keep_tail(make_prompt(q), MAX_PROMPT_TOKENS)],
                apply_chat_template=False,
                mode="eval",
                micro_batch_size=1,
            )
            print("Debug type:", type(dbg))
            if hasattr(dbg, "tokens"):
                print("Debug tokens (first row):", list(map(int, dbg.tokens[0][:min(16, dbg.tokens.shape[1])])))
        except Exception as e:
            print("Debug generation failed:", repr(e))
    else:
        print("A:", ans)

    print("⏱️ one-example time:", round(time.time() - t1, 2), "sec")

print("\n✅ Cell 13 done.")
print("⏱️ total time:", round(time.time() - t0, 2), "sec")

gc.collect()
try:
    jax.clear_caches()
except Exception:
    pass


Backend: cpu | Devices: [CpuDevice(id=0)]
✅ jax_disable_jit=True (best-effort)
✅ Using MAX_PROMPT_TOKENS=48 | MAX_NEW_TOKENS=16 | RUN_N=2
Q: Summarize in 2 sentences: Reinforcement learning improves behavior using rewards.
A: Reinforcement learning (RL) is an approach to training agents to learn optimal behavior
⏱️ one-example time: 291.83 sec
Q: Write a Python function to check palindrome. Mention complexity.
A: ```python
def is_palindrome(text):
  """
  Checks
⏱️ one-example time: 303.09 sec

✅ Cell 13 done.
⏱️ total time: 594.93 sec


## 14) Checkpoints + Multi-session resume (optional 15 pts)
- Your checkpoints are in `/kaggle/working/ckpts`
- To resume in a new session: load latest checkpoint and continue training.
- At the end of your notebook, print your final Kaggle Model name/ID if you publish it.

In [5]:
import os
actor_dir = os.path.join(CFG.ckpt_dir, "actor")
print("actor_dir exists:", os.path.exists(actor_dir))
if os.path.exists(actor_dir):
    print("actor_dir contents:", os.listdir(actor_dir)[:50])


actor_dir exists: True
actor_dir contents: []


In [4]:
import os, glob

print("Checkpoint dir:", CFG.ckpt_dir)

if not os.path.exists(CFG.ckpt_dir):
    print("❌ Checkpoint dir does not exist yet.")
else:
    files = sorted(os.listdir(CFG.ckpt_dir))
    print("Num entries:", len(files))
    print("First 30 entries:", files[:30])

    # Optional: try to locate "latest" checkpoint-ish directory
    candidates = []
    for p in glob.glob(os.path.join(CFG.ckpt_dir, "**"), recursive=True):
        if os.path.isdir(p):
            candidates.append(p)

    if candidates:
        latest = max(candidates, key=lambda p: os.path.getmtime(p))
        print("✅ Most recently modified checkpoint path:", latest)
    else:
        print("⚠️ No subdirectories found under ckpt_dir (might not have saved a ckpt yet).")



Checkpoint dir: /kaggle/working/ckpts
Num entries: 1
First 30 entries: ['actor']
✅ Most recently modified checkpoint path: /kaggle/working/ckpts/


## 15) 3-minute Video Script Outline (for 20 pts)
1) Problem: models don’t show work consistently
2) Approach: Gemma + Tunix + GRPO
3) Data: GSM8K + cross-domain prompts
4) Rewards: format + reasoning quality + correctness
5) Results: before/after samples + metrics
6) How to reproduce: run notebook, load checkpoints
